# Find the Right Care: v5 pipeline and `find_care`

**What this does.** You give a **service** and a **location** (PIN code, district, city or `lat,lon`). The notebook returns the
recommended facilities. Every claim is a verbatim quote from the facility's own record, with an evidence label, a confidence word,
the score breakdown (Evidence / Capability / Access), caveats, what the record doesn't report, and contact details.

**How to use it (no coding needed).**
1. The first time, choose **Run all**. This builds the tables (a few minutes on Free Edition).
2. After that, set **rebuild = no** at the top, pick a **service**, type a **location**, and run the cell **"Your search"** (or Run all).

Method: `Datathon26/docs/PLAN_V5.md` (authoritative), `FEATURES.md`, `SCORING.md`. Every deviation found at run time is printed as
`ADAPT:` and collected in `ADAPTATIONS`. The optional LLM second reader (PLAN_V5 §3) is **off**.

## 0. Settings and widgets

In [0]:
import json, math, re, time, uuid, html, datetime as dt
from pyspark.sql import functions as F, types as T

SERVICES = ["dialysis", "emergency_csection", "nicu", "cardiac_surgery", "mri", "trauma", "burn_care", "oncology", "ent"]
SERVICE_PHRASE = {"dialysis": "dialysis", "emergency_csection": "emergency C-section", "nicu": "newborn intensive care (NICU)",
                  "cardiac_surgery": "cardiac surgery", "mri": "MRI", "trauma": "trauma care", "burn_care": "burn care",
                  "oncology": "cancer care", "ent": "ENT care"}
SERVICE_NAMES = {"dialysis": "Dialysis", "emergency_csection": "Emergency C-section", "nicu": "NICU (newborn intensive care)",
                 "cardiac_surgery": "Cardiac surgery", "mri": "MRI scan", "trauma": "Trauma care", "burn_care": "Burn care (beta)",
                 "oncology": "Cancer care (oncology)", "ent": "ENT (ear, nose, throat)"}

try:
    dbutils  # noqa: F821
    IS_DATABRICKS = True
except NameError:
    IS_DATABRICKS = False

if IS_DATABRICKS:
    dbutils.widgets.dropdown("service", "dialysis", SERVICES, "1. Service")
    dbutils.widgets.text("location", "492001", "2. Location (PIN, district, city or lat,lon)")
    dbutils.widgets.dropdown("k", "5", [str(i) for i in range(1, 11)], "3. How many results")
    dbutils.widgets.dropdown("rebuild", "yes", ["yes", "no"], "4. Rebuild tables")
    dbutils.widgets.dropdown("run_tests", "yes", ["yes", "no"], "5. Run tests and smoke queries")
    W = {k: dbutils.widgets.get(k) for k in ("service", "location", "k", "rebuild", "run_tests")}
else:
    W = dict(globals().get("LOCAL_WIDGETS", {"service": "dialysis", "location": "492001", "k": "5", "rebuild": "yes", "run_tests": "yes"}))

SCHEMA = globals().get("SCHEMA_OVERRIDE", "workspace.hackathon")
OWNER_TAG = "find_care_v5"        # table property ftrc.owner: marks tables this notebook created and may overwrite
REF_YEAR = dt.date.today().year
RUN = {"t0": time.time(), "steps": {}}
ADAPTATIONS = []

def note(msg):
    if msg not in ADAPTATIONS:
        ADAPTATIONS.append(msg)
        print("ADAPT:", msg)

class step:
    """Times a build step: `with step('name'):`."""
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); print(f"▶ {self.name}"); return self
    def __exit__(self, *exc):
        RUN["steps"][self.name] = round(time.time() - self.t, 1)
        print(f"  ✓ {self.name}: {RUN['steps'][self.name]} s")

def q(sql, **args):
    """spark.sql with named parameter markers (regexes are passed as :args, never pasted into SQL text)."""
    return spark.sql(sql, args=args) if args else spark.sql(sql)

print("Databricks:", IS_DATABRICKS, "| schema:", SCHEMA, "| widgets:", W)

## 0.1 Output table names (never overwrite a table this notebook didn't create)

In [0]:
OUTPUT_TABLES = ["silver_facility_clean", "silver_facility_location", "silver_claims", "ref_feature_terms",
                 "ref_feature_context", "ref_feature_specialty", "ref_service_direct", "ref_regression_fixtures",
                 "silver_feature_hits", "gold_facility_features", "gold_facility_completeness", "gold_service_evidence",
                 "ref_service_weights", "gold_service_depth", "gold_service_scores", "ref_service_profile", "gold_query_log"]

def table_exists(fq):
    try:
        return spark.catalog.tableExists(fq)
    except Exception:
        return False

def owned_by_us(fq):
    if not table_exists(fq):
        return None
    try:
        props = {r[0]: r[1] for r in spark.sql(f"SHOW TBLPROPERTIES {fq}").collect()}
    except Exception:
        return False
    return props.get("ftrc.owner") == OWNER_TAG

TBL = {}
for base in OUTPUT_TABLES:
    for cand in (base, base + "_v5"):
        fq = f"{SCHEMA}.{cand}"
        own = owned_by_us(fq)
        if own is None or own:
            TBL[base] = fq
            if cand != base:
                note(f"{SCHEMA}.{base} already exists and wasn't created by this notebook: writing {fq} instead")
            break
    else:
        raise RuntimeError(f"Both {base} and {base}_v5 exist and are not ours. Refusing to overwrite.")
for k_, v_ in TBL.items():
    print(f"  {k_:28s} → {v_}")

def save(base, df):
    """Overwrite one of OUR tables (TBL guarantees ownership) and tag it."""
    fq = TBL[base]
    (df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq))
    spark.sql(f"ALTER TABLE {fq} SET TBLPROPERTIES ('ftrc.owner' = '{OWNER_TAG}')")
    return fq

REBUILD = W["rebuild"] == "yes" or not all(table_exists(TBL[b]) for b in ("gold_service_scores", "silver_facility_location"))
print("Rebuild offline tables:", REBUILD)

## 1. Profile the inputs and confirm the input contract (read-only)

In [0]:
SILVER = f"{SCHEMA}.silver_facilities"
PINS = f"{SCHEMA}.silver_pincode_directory"
SPECMAP = f"{SCHEMA}.ref_specialty_mapping"
BRONZE_PINS = f"{SCHEMA}.bronze_pincode_directory"

sf_types = dict(spark.table(SILVER).dtypes)
pin_types = dict(spark.table(PINS).dtypes)
print(f"silver_facilities: {spark.table(SILVER).count():,} rows, {len(sf_types)} columns")
print(f"silver_pincode_directory: {spark.table(PINS).count():,} rows, columns {sorted(pin_types)}")

def resolve(types, logical, candidates, kind, required=False):
    """Pick the first existing column for a logical field and return a SQL expression of the right type."""
    for c in candidates:
        if c in types:
            if c != candidates[0]:
                note(f"input contract: '{logical}' read from column `{c}` (plan assumes `{candidates[0]}`)")
            t, col = types[c], f"`{c}`"
            if kind == "arr":
                if t.startswith("array"):
                    return f"filter({col}, x -> x IS NOT NULL AND trim(x) <> '')"
                note(f"input contract: `{c}` is {t}, parsed as a JSON array")
                return f"filter(from_json({col}, 'array<string>'), x -> x IS NOT NULL AND trim(x) <> '')"
            if kind == "num":
                return f"try_cast({col} AS DOUBLE)"
            if kind == "bool":
                return f"coalesce(try_cast({col} AS BOOLEAN), false)" if t != "boolean" else f"coalesce({col}, false)"
            if kind == "pin":
                if t in ("int", "bigint", "smallint", "double", "float") or t.startswith("decimal"):
                    return f"try_cast({col} AS BIGINT)"
                return f"try_cast(regexp_replace(trim(CAST({col} AS STRING)), '[^0-9]', '') AS BIGINT)"
            if kind == "raw":
                return f"to_json({col})" if t.startswith("array") else f"CAST({col} AS STRING)"
            return f"nullif(trim(CAST({col} AS STRING)), '')"
    if required:
        raise RuntimeError(f"input contract: none of {candidates} found for '{logical}'")
    note(f"input contract: no column for '{logical}' (tried {candidates}); treated as missing")
    return {"arr": "CAST(NULL AS ARRAY<STRING>)", "num": "CAST(NULL AS DOUBLE)", "bool": "false"}.get(kind, "CAST(NULL AS STRING)")

C = {
    "name":          resolve(sf_types, "name", ["clean_name", "name"], "str", True),
    "facility_type": resolve(sf_types, "facility_type", ["clean_facility_type", "facilityTypeId"], "str"),
    "operator_type": resolve(sf_types, "operator_type", ["clean_operator_type", "operatorTypeId"], "str"),
    "affiliations":  resolve(sf_types, "affiliations", ["clean_affiliations", "affiliationTypeIds"], "arr"),
    "n_doctors":     resolve(sf_types, "n_doctors", ["clean_number_doctors", "clean_numberDoctors", "numberDoctors"], "num"),
    "capacity":      resolve(sf_types, "capacity", ["clean_capacity", "capacity"], "num"),
    "capacity_text": resolve(sf_types, "capacity_text", ["clean_capacity_from_text"], "num"),
    "year":          resolve(sf_types, "year_established", ["yearEstablished", "clean_year_established", "clean_yearEstablished"], "num"),
    "specialties":   resolve(sf_types, "specialties", ["clean_specialties", "specialties"], "arr"),
    "procedures":    resolve(sf_types, "procedures", ["clean_procedures", "clean_procedure", "procedure"], "arr"),
    "equipment":     resolve(sf_types, "equipment", ["clean_equipment", "equipment"], "arr"),
    "capabilities":  resolve(sf_types, "capabilities", ["clean_capabilities", "clean_capability", "capability"], "arr"),
    "description":   resolve(sf_types, "description", ["clean_description", "description"], "str"),
    "flag_doctors":  resolve(sf_types, "flag_numberDoctors_implausible", ["flag_numberDoctors_implausible"], "bool"),
    "flag_coords":   resolve(sf_types, "flag_coordinates_far_from_pincode", ["flag_coordinates_far_from_pincode"], "bool"),
    "pincode":       resolve(sf_types, "pincode", ["clean_pincode", "address_zipOrPostcode"], "pin"),
    "lat":           resolve(sf_types, "latitude", ["clean_latitude", "latitude"], "num"),
    "lon":           resolve(sf_types, "longitude", ["clean_longitude", "longitude"], "num"),
    "phones":        resolve(sf_types, "phone_numbers", ["clean_phone_numbers", "phone_numbers"], "arr"),
    "official_phone": resolve(sf_types, "official_phone", ["clean_officialPhone", "clean_official_phone", "officialPhone"], "str"),
    "last_updated":  resolve(sf_types, "recency_of_page_update", ["clean_recency_of_page_update", "recency_of_page_update"], "str"),
    "district":      resolve(sf_types, "district", ["clean_district", "address_district"], "str"),
    "state":         resolve(sf_types, "state", ["clean_state", "address_stateOrRegion"], "str"),
    "city":          resolve(sf_types, "city", ["address_city", "clean_city"], "str"),
    "website":       resolve(sf_types, "official_website", ["clean_officialWebsite", "clean_website", "clean_official_website", "officialWebsite"], "str"),
    "websites":      resolve(sf_types, "websites", ["clean_websites", "websites"], "arr"),
    # raw text fields: quotes are checked against these (the eval bench does the same)
    "raw_description": resolve(sf_types, "raw description", ["description", "clean_description"], "raw"),
    "raw_procedure":   resolve(sf_types, "raw procedure", ["procedure", "clean_procedures"], "raw"),
    "raw_equipment":   resolve(sf_types, "raw equipment", ["equipment", "clean_equipment"], "raw"),
    "raw_capability":  resolve(sf_types, "raw capability", ["capability", "clean_capabilities"], "raw"),
    "raw_specialties": resolve(sf_types, "raw specialties", ["specialties", "clean_specialties"], "raw"),
}
# city: silver has no city column on the real data, so read the raw address_city from bronze_facilities (read-only)
CITY_JOIN = ""
BRONZE_FAC = f"{SCHEMA}.bronze_facilities"
if C["city"] == "CAST(NULL AS STRING)":
    try:
        bt = dict(spark.table(BRONZE_FAC).dtypes)
        if "address_city" in bt and "unique_id" in bt:
            CITY_JOIN = (f"LEFT JOIN (SELECT unique_id, max(nullif(trim(address_city), '')) AS bronze_city "
                         f"FROM {BRONZE_FAC} GROUP BY unique_id) bc USING (unique_id)")
            C["city"] = "bc.bronze_city"
            note("input contract: city read from bronze_facilities.address_city (joined on unique_id); silver has no city column")
    except Exception as ex:
        note(f"input contract: bronze_facilities not readable for city ({str(ex)[:80]}); city names resolve via district only")

# M7 truncation: silver's flags if present, else "a list has 50 items"
TRUNC_FLAGS = [c for c in ("flag_procedure_truncated_50", "flag_equipment_truncated_50", "flag_capability_truncated_50") if c in sf_types]
if len(TRUNC_FLAGS) == 3:
    C["truncated"] = " OR ".join(f"coalesce(try_cast(`{c}` AS BOOLEAN), false)" for c in TRUNC_FLAGS)
else:
    note(f"input contract: truncation flags missing ({TRUNC_FLAGS} found); using 'a list has ≥ 50 items' (M7) instead")
    C["truncated"] = " OR ".join(f"coalesce(size({C[x]}), 0) >= 50" for x in ("procedures", "equipment", "capabilities"))

PC = {
    "pin":     resolve(pin_types, "pin pincode", ["pincode", "pin"], "num", True),
    "lat":     resolve(pin_types, "pin latitude", ["latitude", "lat", "avg_latitude"], "num", True),
    "lon":     resolve(pin_types, "pin longitude", ["longitude", "lon", "avg_longitude"], "num", True),
    "primary": resolve(pin_types, "is_primary_state", ["is_primary_state"], "bool"),
    "district": resolve(pin_types, "pin district", ["district", "districtname", "district_name", "clean_district"], "str"),
    "state":   resolve(pin_types, "pin state", ["statename", "state", "state_name", "clean_state"], "str"),
}
if "is_primary_state" not in pin_types:
    PC["primary"] = "true"

# PIN coordinate guard. Some PIN points in the directory sit in the wrong state (19 of the 25 Aurangabad, Bihar PINs are
# geocoded to Aurangabad, Maharashtra), probably from geocoding by district name. A PIN point more than PIN_OUTLIER_KM from
# the median of its 3-digit PIN prefix (one sorting area) is an outlier: its lat/lon is NULL here, lat_pre/lon_pre hold the
# prefix median. The input table is not modified.
PIN_OUTLIER_KM = 300   # 300 catches all 19 bad Aurangabad (Bihar) points and spares real far-flung PINs such as Car Nicobar (277 km)
PINS_OK = f"""(
  WITH pp AS (SELECT {PC['pin']} AS pin, {PC['lat']} AS lat_dir, {PC['lon']} AS lon_dir, {PC['district']} AS district,
                     {PC['state']} AS state, CAST(floor({PC['pin']} / 1000) AS BIGINT) AS pre
              FROM {PINS} WHERE {PC['primary']}),
  pm AS (SELECT pre, percentile_approx(lat_dir, 0.5) AS lat_pre, percentile_approx(lon_dir, 0.5) AS lon_pre
         FROM pp WHERE lat_dir IS NOT NULL AND lon_dir IS NOT NULL GROUP BY pre),
  pd AS (SELECT pp.*, pm.lat_pre, pm.lon_pre,
           2 * 6371 * asin(sqrt(pow(sin(radians(pp.lat_dir - pm.lat_pre) / 2), 2)
             + cos(radians(pp.lat_dir)) * cos(radians(pm.lat_pre)) * pow(sin(radians(pp.lon_dir - pm.lon_pre) / 2), 2))) AS km_from_prefix
         FROM pp LEFT JOIN pm ON pm.pre = pp.pre)
  SELECT pin, district, state, lat_dir, lon_dir, lat_pre, lon_pre,
         coalesce(km_from_prefix > {PIN_OUTLIER_KM}, false) AS coord_outlier,
         CASE WHEN coalesce(km_from_prefix <= {PIN_OUTLIER_KM}, true) THEN lat_dir END AS lat,
         CASE WHEN coalesce(km_from_prefix <= {PIN_OUTLIER_KM}, true) THEN lon_dir END AS lon
  FROM pd)"""
_po = q(f"SELECT count(*) AS n, count_if(coord_outlier) AS outliers FROM {PINS_OK}").collect()[0]
if _po["outliers"]:
    note(f"PIN directory: {_po['outliers']} of {_po['n']} PIN points are > {PIN_OUTLIER_KM} km from their 3-digit-prefix median; "
         "treated as unknown for facility locations and district centres (input not modified)")

# --- value profiling -------------------------------------------------------------------------------------------
prof = q(f"""
SELECT count(*) AS n,
  count_if({C['lat']} IS NOT NULL) AS has_lat,
  count_if({C['flag_coords']}) AS flag_coords,
  count_if({C['flag_doctors']}) AS flag_doctors,
  count_if({C['truncated']}) AS truncated,
  count_if({C['n_doctors']} IS NOT NULL) AS has_doctors,
  count_if(coalesce({C['capacity']}, {C['capacity_text']}) IS NOT NULL) AS has_beds,
  count_if(size({C['specialties']}) > 0) AS has_specialties
FROM {SILVER}""").collect()[0].asDict()
print("profile:", prof)
print("facility types:", {r[0]: r[1] for r in q(f"SELECT {C['facility_type']} t, count(*) n FROM {SILVER} GROUP BY 1 ORDER BY 2 DESC").collect()})

# Specialty codes the method relies on (SCORING §3.1, FEATURES §4.3, PLAN_V5 R1/R4/X7)
NEEDED_CODES = ["nephrology", "obstetrics", "maternalFetalMedicine", "gynecology", "neonatology", "pediatricCriticalCare",
                "pediatrics", "cardiothoracicSurgery", "cardiology", "radiology", "emergencyMedicine", "burnCare",
                "medicalOncology", "surgicalOncology", "radiationOncology", "oncology", "otolaryngology", "anesthesiology",
                "criticalCareMedicine", "nuclearMedicine", "audiology"]
code_counts = {r[0]: r[1] for r in q(f"SELECT s, count(DISTINCT unique_id) FROM (SELECT unique_id, explode({C['specialties']}) s FROM {SILVER}) GROUP BY s").collect()}
print(f"distinct specialty codes in silver: {len(code_counts)}")
CODE_OK = {}
for c in NEEDED_CODES:
    if c in code_counts:
        CODE_OK[c] = True
        print(f"  ✓ {c:24s} {code_counts[c]:>6,} facilities")
    else:
        CODE_OK[c] = False
        sugg = sorted((k for k in code_counts if c.lower()[:6] in k.lower()), key=lambda k: -code_counts[k])[:5]
        print(f"  ✗ {c:24s} not in silver. Similar: {sugg}")
        note(f"specialty code `{c}` not found in silver_facilities (similar: {sugg}); rules using it can't fire")

try:
    print("ref_specialty_mapping columns:", spark.table(SPECMAP).dtypes)
except Exception as e:
    note(f"ref_specialty_mapping not readable: {e}")

# Quote fidelity: is clean text a verbatim substring of the raw field the eval bench checks?
qf = q(f"""
WITH x AS (SELECT {C['raw_procedure']} AS raw, explode({C['procedures']}) AS item FROM {SILVER})
SELECT count(*) AS items, count_if(raw IS NOT NULL AND instr(raw, item) > 0) AS verbatim_in_raw FROM x""").collect()[0]
print(f"procedure items verbatim in raw field: {qf.verbatim_in_raw:,}/{qf['items']:,}")
if qf["items"] and qf.verbatim_in_raw < qf["items"]:
    note(f"{qf['items'] - qf.verbatim_in_raw:,} clean procedure items are not byte-identical substrings of the raw JSON field "
         "(usually JSON escaping); quotes are re-checked against the decoded raw list at build time")

## 2. Step 1: `silver_facility_clean` and `silver_facility_location` (FEATURES §4.1, M1/M3/M5/M7/M8, G15)

In [0]:
NAME_TYPE_RX = [  # FEATURES §3.2 G15, order matters
    ("nephrology_dialysis", r"kidney|dialysis|nephro|renal"),
    ("maternity", r"maternity|prasuti|mother|women.?s hospital|birthing"),
    ("oncology", r"cancer|onco"),
    ("cardiac", r"heart|cardiac|cardio"),
    ("trauma", r"trauma|accident"),
    ("eye", r"eye|netra|nethra|nayan|ophthal|vision"),
    ("dental", r"dental|dentist|teeth|smile|orthodont"),
    ("diagnostic", r"diagnostic|imaging|scan|patholog|\blab"),
]

if REBUILD:
    with step("1 silver_facility_clean"):
        name_case = "CASE " + " ".join(f"WHEN lower(n) rlike :nt{i} THEN '{t}'" for i, (t, _) in enumerate(NAME_TYPE_RX)) + " END"
        df = q(f"""
        WITH b AS (
          SELECT unique_id,
            {C['name']} AS n, {C['facility_type']} AS facility_type, {C['operator_type']} AS operator_type,
            {C['affiliations']} AS affiliations,
            CASE WHEN NOT {C['flag_doctors']} THEN {C['n_doctors']} END AS n_doctors,            -- M8
            {C['capacity']} AS capacity, {C['capacity_text']} AS capacity_text,
            try_cast({C['year']} AS INT) AS year_established,
            {C['specialties']} AS sp, {C['procedures']} AS pr, {C['equipment']} AS eq, {C['capabilities']} AS ca,
            {C['description']} AS description_ok,
            ({C['truncated']}) AS truncated,
            {C['pincode']} AS pincode,
            CASE WHEN NOT {C['flag_coords']} THEN {C['lat']} END AS lat_raw,                         -- M8
            CASE WHEN NOT {C['flag_coords']} THEN {C['lon']} END AS lon_raw,
            {C['phones']} AS phone_numbers, {C['official_phone']} AS official_phone,
            {C['last_updated']} AS last_updated, {C['district']} AS district, {C['state']} AS state, {C['city']} AS city,
            {C['website']} AS official_website, {C['websites']} AS websites,
            {C['raw_description']} AS raw_description, {C['raw_procedure']} AS raw_procedure,
            {C['raw_equipment']} AS raw_equipment, {C['raw_capability']} AS raw_capability,
            {C['raw_specialties']} AS raw_specialties
          FROM {SILVER} {CITY_JOIN})
        SELECT unique_id, n AS name, facility_type, operator_type, affiliations, n_doctors, capacity, capacity_text, year_established,
          CASE WHEN size(sp) > 0 THEN sp END AS specialties_arr,                                       -- M1
          CASE WHEN size(pr) > 0 THEN pr END AS procedure_arr,
          CASE WHEN size(eq) > 0 THEN eq END AS equipment_arr,
          CASE WHEN size(ca) > 0 THEN ca END AS capability_arr,
          description_ok,
          coalesce(size(pr), 0) > 0 OR coalesce(size(eq), 0) > 0 OR coalesce(size(ca), 0) > 0 OR description_ok IS NOT NULL AS has_text,  -- M3
          coalesce(truncated, false) AS truncated,                                                    -- M7
          {name_case} AS name_type,                                                                   -- G15
          pincode, lat_raw, lon_raw, phone_numbers, official_phone, last_updated, district, state, city,
          official_website, websites, raw_description, raw_procedure, raw_equipment, raw_capability, raw_specialties
        FROM b""", **{f"nt{i}": rx for i, (_, rx) in enumerate(NAME_TYPE_RX)})
        save("silver_facility_clean", df)

    with step("1 silver_facility_location"):
        has_bronze = table_exists(BRONZE_PINS)
        if not has_bronze:
            note("bronze_pincode_directory not readable: PIN radius defaults to 5 km for approximate locations")
        pin_radius_cte = f"""
          po AS (SELECT try_cast(pincode AS BIGINT) AS pin, try_cast(latitude AS DOUBLE) AS la, try_cast(longitude AS DOUBLE) AS lo
                 FROM {BRONZE_PINS}),
          pr AS (SELECT po.pin, percentile_approx(2 * 6371 * asin(sqrt(pow(sin(radians(po.la - p.lat) / 2), 2)
                    + cos(radians(p.lat)) * cos(radians(po.la)) * pow(sin(radians(po.lo - p.lon) / 2), 2))), 0.9) AS pin_radius_km
                 FROM po JOIN p ON p.pin = po.pin WHERE po.la IS NOT NULL AND po.lo IS NOT NULL GROUP BY po.pin),""" if has_bronze else \
            "pr AS (SELECT CAST(NULL AS BIGINT) AS pin, CAST(NULL AS DOUBLE) AS pin_radius_km),"
        df = q(f"""
        WITH p AS (SELECT pin, lat, lon FROM {PINS_OK} WHERE lat IS NOT NULL),
        {pin_radius_cte}
        f AS (SELECT unique_id, pincode, lat_raw, lon_raw FROM {TBL['silver_facility_clean']})
        SELECT f.unique_id, f.pincode,
          CASE WHEN f.lat_raw IS NOT NULL AND f.lon_raw IS NOT NULL THEN f.lat_raw ELSE p.lat END AS lat,
          CASE WHEN f.lat_raw IS NOT NULL AND f.lon_raw IS NOT NULL THEN f.lon_raw ELSE p.lon END AS lon,
          CASE WHEN f.lat_raw IS NOT NULL AND f.lon_raw IS NOT NULL THEN 'exact'
               WHEN p.lat IS NOT NULL THEN 'approximate' ELSE 'unknown' END AS location_quality,      -- M5
          round(least(30.0, greatest(1.0, coalesce(pr.pin_radius_km, 5.0))), 1) AS pin_radius_km
        FROM f LEFT JOIN p ON p.pin = f.pincode LEFT JOIN pr ON pr.pin = f.pincode""")
        save("silver_facility_location", df)
    print({r[0]: r[1] for r in q(f"SELECT location_quality, count(*) FROM {TBL['silver_facility_location']} GROUP BY 1").collect()})

## 3. Step 2: `silver_claims` (FEATURES §4.2 + X3 negation + X6 clause dedup key)

In [0]:
SPLIT_RX = r";|\.\s|,\s*(?=but |however|except)"
NEG_RX = (r"^\s*(no|not|none)\b"
          r"|\bnot (available|provided|mentioned|specified|listed|offered)\b"
          r"|\bno (explicit|specific|information|details|data)\b"
          r"|referr?ed (to|out)|refers? (patients )?to"
          r"|outsourc|under construction"
          r"|will be (installed|started|available)|coming soon|upcoming"
          r"|proposed|(is|are) planned|planned to"
          r"|\bnon[- ]?(nabh|jci|nabl|accredited|functional|operational)\b")          # X3

def claims_df(clean_view):
    """One row per clause. Used for the real build and for the regression fixtures (same code path)."""
    return q(f"""
    WITH u AS (
      SELECT unique_id, 'procedure' AS field, pos, c FROM {clean_view} LATERAL VIEW posexplode(procedure_arr) t AS pos, c
      UNION ALL SELECT unique_id, 'equipment', pos, c FROM {clean_view} LATERAL VIEW posexplode(equipment_arr) t AS pos, c
      UNION ALL SELECT unique_id, 'capability', pos, c FROM {clean_view} LATERAL VIEW posexplode(capability_arr) t AS pos, c
      UNION ALL SELECT unique_id, 'description', 0, description_ok FROM {clean_view} WHERE description_ok IS NOT NULL),
    cl AS (
      SELECT unique_id, field, pos, c AS item, cpos, trim(clause) AS clause
      FROM u LATERAL VIEW posexplode(split(c, :split_rx)) t AS cpos, clause)
    SELECT unique_id, field, pos, cpos,
      clause                                          AS text,
      lower(clause)                                   AS lc,
      lower(item)                                     AS item_lc,
      regexp_replace(lower(item), '[^a-z0-9]', '')    AS norm_item,
      regexp_replace(lower(clause), '[^a-z0-9]', '')  AS norm_clause,     -- X6 dedup key
      lower(clause) rlike :neg_rx                     AS is_negated
    FROM cl
    WHERE clause IS NOT NULL AND clause <> ''""", split_rx=SPLIT_RX, neg_rx=NEG_RX)

if REBUILD:
    with step("2 silver_claims"):
        save("silver_claims", claims_df(TBL["silver_facility_clean"]))
    print(q(f"SELECT count(*) AS clauses, count_if(is_negated) AS negated FROM {TBL['silver_claims']}").collect()[0].asDict())

## 4. Step 3: reference tables (FEATURES §4.3, SCORING §3.1, with X1, X2, X4, X5, X7)

In [0]:
# feature, term_id, kind, rx, num_rx, num_rx2, num_rx3, not_rx, clause_ctx_rx, clause_block_rx, qty_min, qty_max, qty_ref
# Java regex (Spark rlike), applied to the lower-cased clause.
DIALYSIS_X2_FWD = r"dialys[^.;]{0,40}?\b\d+\s*(units?|machines?|beds?|stations?|chairs?)\b"
DIALYSIS_X2_REV = r"\b\d+\s*-?\s*(bedded\s*)?(units?|machines?|beds?|stations?|chairs?)\b[^.;]{0,30}?dialys"
FEATURE_TERMS = [
 ("beds", 1, "qty_only", r"\d+\s*-?\s*(bedded|beds?)\b", r"(\d+)\s*-?\s*(?:bedded|beds?)\b", None, None,
  r"icu|nicu|picu|dialys|ward|private|semi", None, None, 5.0, 5000.0, 1000.0),
 ("icu", 1, "presence_qty", r"\bicu\b|intensive care|intensive critical care",                 # X5: expanded ICCU counts as G4
  r"(\d+)\s*-?\s*(?:bedded|beds?)?\s*(?:icu|intensive)", None, None, None, None, None, 1.0, 200.0, 20.0),
 # X4: emergency phrases always count; 24x7 only with an emergency word and never next to pharmacy/nursing/OPD/lab...
 ("emergency_24x7", 1, "presence",
  r"emergency (and [a-z]+ )?(department|services?|care|ward|unit|room)|casualty|accident (and|&) emergency|trauma (care|centre|center)",
  None, None, None, None, None, None, None, None, None),
 ("emergency_24x7", 2, "presence", r"24\s*x\s*7|24/7|24 hours|round[- ]the[- ]clock", None, None, None, None,
  r"emergenc|casualty|trauma|labou?r|deliver",
  r"\b(pharmacy|nursing|opd|lab|laborator(y|ies)|diagnostics?|ambulances?|chemists?|canteen)\b", None, None, None),
 ("ambulance", 1, "presence", r"ambulance", None, None, None, None, None, None, None, None, None),
 ("ot", 1, "presence_qty", r"operation theat|operating theat|\bmodular ot|\bots?\b",
  r"(\d+)\s*(?:modular\s*)?(?:operation|operating|ots?\b)", None, None, None, None, None, 1.0, 60.0, 10.0),
 ("blood_bank", 1, "presence", r"blood bank|blood storage|blood centre", None, None, None, None, None, None, None, None, None),
 ("anaesthesia", 1, "presence", r"anaesthe|anesthe", None, None, None, None, None, None, None, None, None),
 ("acc_jci", 1, "presence", r"\bjci\b|joint commission", None, None, None, None, None, None, None, None, None),
 ("acc_nabh", 1, "presence", r"\bnabh\b", None, None, None, None, None, None, None, None, None),
 ("lab_accreditation", 1, "presence", r"\bnabl\b", None, None, None, None, None, None, None, None, None),
 ("pmjay", 1, "presence", r"ayushman|pm-?jay", None, None, None, None, None, None, None, None, None),
 ("teaching_text", 1, "presence", r"medical college|institute of medical sciences|\baiims\b|teaching hospital",
  None, None, None, None, None, None, None, None, None),
 ("xray", 1, "presence", r"x-?ray|radiograph", None, None, None, None, None, None, None, None, None),
 ("usg", 1, "presence", r"ultrasound|ultrasonograph|\busg\b|sonograph", None, None, None, None, None, None, None, None, None),
 ("mammography", 1, "presence", r"mammogra", None, None, None, None, None, None, None, None, None),
 # S1 + X2: quantity within 40 chars after / 30 chars before "dialys"
 ("dialysis_unit", 1, "presence_qty",
  r"dialysis (machines?|units?|stations?|chairs?|beds?|cent(re|er)|wing)|ha?emodialysis machines?|ro (water )?plant|"
  + DIALYSIS_X2_FWD + "|" + DIALYSIS_X2_REV,
  r"(\d+)\s*-?\s*(?:bedded\s*|beds?\s*)?(?:dialysis|haemodialysis|hemodialysis)",
  r"dialys[^.;]{0,40}?\b(\d+)\s*(?:units?|machines?|beds?|stations?|chairs?)\b",
  r"\b(\d+)\s*-?\s*(?:bedded\s*)?(?:units?|machines?|beds?|stations?|chairs?)\b[^.;]{0,30}?dialys",
  None, None, None, 1.0, 100.0, 20.0),
 ("labour_room", 1, "presence", r"labou?r room|delivery room|labou?r ward", None, None, None, None, None, None, None, None, None),
 ("nicu", 1, "presence_qty", r"\bnicu\b|neonatal intensive|\bsncu\b",
  r"(\d+)\s*-?\s*(?:bedded|beds?)?\s*(?:nicu|neonatal|sncu)", None, None, None, None, None, 1.0, 150.0, 20.0),
 ("neonatal_support", 1, "presence", r"ventilator|cpap|warmer|incubator|phototherapy", None, None, None, None, None, None, None, None, None),
 ("cath_lab", 1, "presence", r"cath\s*lab|catheteri[sz]ation lab", None, None, None, None, None, None, None, None, None),
 # X5: no plain "cardiac care"; an ICCU expanded to "intensive critical care" is G4, not S6
 ("ccu", 1, "presence", r"\bccu\b|\biccu\b|coronary care|cardiac (intensive|critical) care|\bcicu\b|\bcticu\b",
  None, None, None, None, None, r"intensive critical care", None, None, None),
 ("mri", 1, "presence_qty", r"\bmri\b|magnetic resonance", r"(\d(?:\.\d)?)\s*-?\s*t(?:esla)?\b", None, None, None, None, None, 0.2, 7.0, 3.0),
 ("ct", 1, "presence_qty", r"ct scan|\bct\b|computed tomograph", r"(\d+)\s*-?\s*slice", None, None, None, None, None, 1.0, 640.0, 128.0),
 ("pet", 1, "presence", r"pet[- ]?ct|pet scan", None, None, None, None, None, None, None, None, None),
 ("radiotherapy", 1, "presence", r"radiotherap|radiation (therapy|oncology)|linac|linear accelerator|brachytherap",
  None, None, None, None, None, None, None, None, None),
 ("chemotherapy", 1, "presence", r"chemotherap", None, None, None, None, None, None, None, None, None),
 ("audiometry", 1, "presence", r"audiometr|cochlear|\bbera\b|tympanometr", None, None, None, None, None, None, None, None, None),
 # R5: clinic-hours conflict note (a flag, not part of D)
 ("clinic_hours", 1, "flag", r"opd timings?|clinic hours?|\d{1,2}(:\d{2})?\s*(am|pm)\s*(to|-)\s*\d{1,2}(:\d{2})?\s*(am|pm)",
  None, None, None, None, None, None, None, None, None),
]
TERMS_SCHEMA = ("feature STRING, term_id INT, kind STRING, rx STRING, num_rx STRING, num_rx2 STRING, num_rx3 STRING, not_rx STRING, "
                "clause_ctx_rx STRING, clause_block_rx STRING, qty_min DOUBLE, qty_max DOUBLE, qty_ref DOUBLE")
FEATURE_CONTEXT = [("neonatal_support", r"neonat|newborn|infant|nicu|sncu|baby|paediatric|pediatric")]
FEATURE_SPECIALTY = [("emergency_24x7", "emergencyMedicine"), ("anaesthesia", "anesthesiology"), ("icu", "criticalCareMedicine"),
                     ("nicu", "neonatology"), ("radiotherapy", "radiationOncology"), ("chemotherapy", "medicalOncology"),
                     ("pet", "nuclearMedicine"), ("audiometry", "audiology")]
S_FEATURES = ["dialysis_unit", "labour_room", "nicu", "neonatal_support", "cath_lab", "ccu", "mri", "ct", "pet",
              "radiotherapy", "chemotherapy", "audiometry"]

# ---- service evidence definitions (SCORING §3.1 + X1, X2, X7, R1, R2). Python regex, lower-cased clause. ------------------
# direct: names the service · strip: equipment phrase removed before the direct test (§3.2 step 4)
# equip: service-specific equipment for corroboration / Strong (optionally needing equip_ctx in the same item)
# cond/cond_item/cond_codes: X1 plain "bypass surgery" · excl: same-clause exclusion · codes: specialty codes (exact)
# specialist: G15 name type that lets a description/capability claim with a quantity be Strong (R2)
SVC = {
 "dialysis": dict(direct=r"dialys|\bcrrt\b",
    strip=r"dialysis (?:machines?|units?|stations?|chairs?|beds?)|ha?emodialysis machines?|ro (?:water )?plant",
    equip=r"dialysis (?:machines?|units?|stations?|chairs?|beds?)|ha?emodialysis machines?|ro (?:water )?plant|nephrologist|"
          r"dialys[^.;]{0,40}?\b\d+\s*(?:units?|machines?|beds?|stations?|chairs?)\b|"
          r"\b\d+\s*-?\s*(?:bedded\s*)?(?:units?|machines?|beds?|stations?|chairs?)\b[^.;]{0,30}?dialys",
    codes=["nephrology"], specialist="nephrology_dialysis", implied=[], checked=["nephrology"]),
 "emergency_csection": dict(direct=r"caesarean|cesarean|c-section|\blscs\b",
    equip=r"operation theat|operating theat|\bmodular ot|labou?r room|delivery room|anaesthe|anesthe|\bnicu\b",
    codes=["obstetrics", "maternalFetalMedicine"], specialist="maternity", implied=["ot", "anaesthesia"], checked=["blood_bank"]),
 "nicu": dict(direct=r"\bnicu\b|neonatal intensive",
    equip=r"ventilator|cpap|warmer|incubator|phototherapy|\bsncu\b|level[- ]?(?:ii|2|iii|3)\b",
    equip_ctx=r"neonat|newborn|infant|nicu|sncu|baby|paediatric|pediatric",
    codes=["neonatology", "pediatricCriticalCare"], specialist=None, implied=[], checked=["neo_code"]),
 "cardiac_surgery": dict(
    direct=r"\bcabg\b|coronary (?:artery )?bypass|open[- ]heart|valve (?:replacement|repair)|cardiac surg|cardiothoracic surg|(?:cardiac|heart) bypass",
    cond=r"bypass surgery", cond_item=r"cardi|heart|coronary|cabg|cath", cond_codes=["cardiothoracicSurgery", "cardiology"],
    excl=r"gastric|bariatric|obesity|weight loss|roux",
    equip=r"cath\s*lab|catheteri[sz]ation lab|\bcticu\b|\bcicu\b|heart[- ]lung",
    codes=["cardiothoracicSurgery"], specialist="cardiac", implied=["ot", "anaesthesia", "icu_ccu"], checked=["blood_bank"]),
 "mri": dict(direct=r"\bmri\b|magnetic resonance", equip=None, codes=["radiology"], specialist=None,
    implied=[], checked=["radiology_code"]),
 "trauma": dict(direct=r"trauma (?:care|centre|center|unit|ward|surgery)|polytrauma|accident (?:and|&) emergency",
    excl=r"eye|ocular|orbit|dental|tooth|teeth|maxillofacial|facial",
    equip=r"ct scan|\bct\b|computed tomograph|ambulance", codes=["emergencyMedicine"], specialist="trauma",
    implied=["emergency_24x7"], checked=["ot"]),
 "burn_care": dict(direct=r"burns? (?:unit|ward|centre|center|icu|care)|burn injur",
    equip=r"burns? (?:icu|intensive)|dressing room", codes=["burnCare"], specialist=None, implied=[], checked=["ot", "icu"]),
 "oncology": dict(direct=r"chemotherap|radiotherap|oncolog|cancer (?:surgery|treatment|care)",
    equip=r"linac|linear accelerator|pet[- ]?ct|brachytherap",
    codes=["medicalOncology", "surgicalOncology", "radiationOncology", "oncology"], specialist="oncology",
    implied=[], checked=["onco_code"]),
 "ent": dict(direct=r"\bent\b|otolaryngolog|tonsillectomy|cochlear implant|sinus surgery",
    equip=r"audiometr|\bbera\b|ent endoscop|tympanometr", codes=["otolaryngology"], specialist=None,
    implied=[], checked=["ent_code"]),
}
PREREQ_NAMES = {"ot": "operating theatre", "anaesthesia": "anaesthesia", "icu_ccu": "ICU or CCU", "blood_bank": "blood bank",
                "emergency_24x7": "24x7 emergency", "nephrology": "nephrology specialty or nephrologist",
                "neo_code": "neonatology or paediatrics specialty", "radiology_code": "radiology specialty",
                "onco_code": "oncology specialty", "ent_code": "ENT specialty", "icu": "ICU"}

# Ranking profiles: SCORING §4.2 (authoritative, R6) with R4's C-section change.
PROFILE = {  # service: (time_critical, gate_min_label, wE, wD, wA, d_half_km, max_radius_km)
    "emergency_csection": (True,  "Moderate", .35, .20, .45, 20, 100),
    "nicu":               (True,  "Moderate", .35, .15, .50, 25, 200),
    "trauma":             (True,  "Moderate", .30, .15, .55, 15, 100),
    "dialysis":           (False, "Moderate", .30, .10, .60, 15, 150),
    "burn_care":          (False, "Moderate", .35, .25, .40, 40, 400),
    "mri":                (False, "Weak",     .40, .10, .50, 50, 300),
    "cardiac_surgery":    (False, "Moderate", .35, .45, .20, 150, 500),
    "oncology":           (False, "Moderate", .35, .45, .20, 150, 500),
    "ent":                (False, "Weak",     .40, .30, .30, 40, 150),
}
RADIUS_STEPS = [25, 50, 100, 250]

# Depth weights: FEATURES §3.3 with R4 (C-section) applied. Each service sums to 1.
WEIGHTS = {
 "dialysis": {"dialysis_unit": .35, "icu": .15, "accreditation": .10, "doctors": .10, "beds": .10, "facility_type": .20},
 "emergency_csection": {"labour_room": .10, "nicu": .10, "emergency_24x7": .15, "ot": .15, "blood_bank": .15,
                        "accreditation": .05, "beds": .10, "obstetric_focus": .20},
 "nicu": {"nicu": .25, "neonatal_support": .20, "emergency_24x7": .15, "accreditation": .15, "teaching": .10, "beds": .15},
 "cardiac_surgery": {"cath_lab": .30, "ccu": .20, "ot": .10, "blood_bank": .05, "accreditation": .15, "teaching": .10, "beds": .10},
 "mri": {"mri": .45, "emergency_24x7": .20, "lab_accreditation": .15, "accreditation": .20},
 "oncology": {"radiotherapy": .30, "chemotherapy": .20, "pet": .15, "accreditation": .10, "teaching": .15, "beds": .10},
 "ent": {"doctors": .35, "audiometry": .25, "ot": .15, "accreditation": .15, "beds": .10},
 "trauma": {"ct": .20, "emergency_24x7": .20, "ot": .15, "blood_bank": .15, "ambulance": .10, "icu": .10, "beds": .10},
 "burn_care": {"icu": .25, "ot": .20, "blood_bank": .15, "beds": .15, "accreditation": .10, "teaching": .15},
}
for s_, w_ in WEIGHTS.items():
    assert abs(sum(w_.values()) - 1) < 1e-9, (s_, sum(w_.values()))
for s_, d_ in SVC.items():   # drop codes that don't exist in silver (reported in §1) so they can't silently "match"
    for key in ("codes", "cond_codes"):
        missing_ = [c for c in d_.get(key, []) if not CODE_OK.get(c, True)]
        if missing_:
            note(f"{s_}: {key} {missing_} absent from silver specialties")

if REBUILD:
    with step("3 reference tables"):
        save("ref_feature_terms", spark.createDataFrame(FEATURE_TERMS, TERMS_SCHEMA))
        save("ref_feature_context", spark.createDataFrame(FEATURE_CONTEXT, "feature STRING, ctx_rx STRING"))
        save("ref_feature_specialty", spark.createDataFrame(FEATURE_SPECIALTY, "feature STRING, specialty STRING"))
        save("ref_service_direct", spark.createDataFrame(
            [(s, d["direct"], d.get("strip"), d.get("equip"), d.get("equip_ctx"), d.get("excl"), d.get("cond"), d.get("cond_item"),
              d.get("cond_codes", []), d["codes"], d.get("specialist"), d["implied"], d["checked"]) for s, d in SVC.items()],
            "service STRING, direct_rx STRING, equip_strip_rx STRING, equip_rx STRING, equip_ctx_rx STRING, excl_rx STRING, "
            "cond_rx STRING, cond_item_rx STRING, cond_codes ARRAY<STRING>, codes ARRAY<STRING>, specialist_name_type STRING, "
            "implied_prereqs ARRAY<STRING>, checked_prereqs ARRAY<STRING>"))
        save("ref_service_weights", spark.createDataFrame(
            [(s, f, float(w)) for s, ws in WEIGHTS.items() for f, w in ws.items()], "service STRING, feature STRING, weight DOUBLE"))
        save("ref_service_profile", spark.createDataFrame(
            [(s, p[0], p[1], *map(float, p[2:])) for s, p in PROFILE.items()],
            "service STRING, time_critical BOOLEAN, gate_min_label STRING, wE DOUBLE, wD DOUBLE, wA DOUBLE, d_half_km DOUBLE, max_radius_km DOUBLE"))

## 5. Feature hits and features (FEATURES §4.4–4.5, X4/X5, G16, R5 flag)

In [0]:
def hits_df(claims_view, clean_view):
    return spark.sql(f"""
    WITH m AS (
      SELECT c.unique_id, t.feature, t.qty_min, t.qty_max, c.field, c.pos, c.cpos, c.text,
        CASE WHEN t.num_rx IS NOT NULL AND (t.not_rx IS NULL OR NOT c.lc rlike t.not_rx) THEN
          try_cast(coalesce(
            nullif(regexp_extract(c.lc, t.num_rx, 1), ''),
            CASE WHEN t.num_rx2 IS NOT NULL THEN nullif(regexp_extract(c.lc, t.num_rx2, 1), '') END,
            CASE WHEN t.num_rx3 IS NOT NULL THEN nullif(regexp_extract(c.lc, t.num_rx3, 1), '') END) AS DOUBLE)
        END AS qty_raw
      FROM {claims_view} c
      JOIN {TBL['ref_feature_terms']} t ON c.lc rlike t.rx
      LEFT JOIN {TBL['ref_feature_context']} x ON x.feature = t.feature
      WHERE NOT c.is_negated                                              -- M6
        AND (x.ctx_rx IS NULL OR c.item_lc rlike x.ctx_rx)                -- context word in the same item
        AND (t.clause_ctx_rx IS NULL OR c.lc rlike t.clause_ctx_rx)       -- X4: 24x7 needs an emergency word in the clause
        AND (t.clause_block_rx IS NULL OR NOT c.lc rlike t.clause_block_rx))
    SELECT unique_id, feature, field, pos, cpos, text,
      CAST(CASE field WHEN 'equipment' THEN 1.0 WHEN 'procedure' THEN 1.0 WHEN 'capability' THEN 0.9 ELSE 0.6 END AS DOUBLE) AS strength,
      CASE WHEN qty_raw BETWEEN qty_min AND qty_max THEN qty_raw END AS qty
    FROM m
    UNION ALL                                                             -- weak specialty-tag hits (0.3)
    SELECT e.unique_id, s.feature, 'specialties', CAST(NULL AS INT), CAST(NULL AS INT), e.sp, CAST(0.3 AS DOUBLE), CAST(NULL AS DOUBLE)
    FROM (SELECT unique_id, explode(specialties_arr) AS sp FROM {clean_view}) e
    JOIN {TBL['ref_feature_specialty']} s ON s.specialty = e.sp""")

def features_df(hits_view, clean_view):
    return spark.sql(f"""
    WITH agg AS (
      SELECT unique_id, feature, max(strength) AS s, max(qty) AS q, max(field <> 'specialties') AS found_in_text,
        slice(array_sort(collect_list(named_struct('strength', strength, 'field', field, 'text', text)),
              (a, b) -> CASE WHEN a.strength > b.strength THEN -1 WHEN a.strength < b.strength THEN 1 ELSE 0 END), 1, 3) AS evidence
      FROM {hits_view} GROUP BY unique_id, feature),
    tf AS (SELECT feature, first(kind) AS kind, first(qty_ref) AS qty_ref FROM {TBL['ref_feature_terms']} GROUP BY feature),
    text_feats AS (
      SELECT f.unique_id, t.feature,
        CASE WHEN a.s IS NULL AND NOT f.has_text THEN NULL                                   -- unknown (M3)
             WHEN a.s IS NULL AND f.truncated THEN NULL                                      -- M7
             WHEN a.s IS NULL THEN 0.0                                                       -- not mentioned
             WHEN t.kind IN ('presence', 'flag') THEN a.s
             WHEN t.kind = 'presence_qty' AND a.q IS NOT NULL
               THEN a.s * (0.6 + 0.4 * CASE WHEN t.feature = 'mri' THEN least(1.0, a.q / t.qty_ref)
                                           ELSE least(1.0, ln(1 + a.q) / ln(1 + t.qty_ref)) END)
             WHEN t.kind = 'presence_qty' THEN a.s * 0.6 END                                  AS value,
        CASE WHEN a.s IS NOT NULL THEN 'found' WHEN NOT f.has_text THEN 'unknown'
             WHEN f.truncated THEN 'truncated' ELSE 'not_mentioned' END                      AS status,
        a.q AS qty, coalesce(a.found_in_text, false) AS found_in_text, a.evidence
      FROM {clean_view} f CROSS JOIN tf t LEFT JOIN agg a ON a.unique_id = f.unique_id AND a.feature = t.feature
      WHERE t.kind <> 'qty_only'),
    h AS (
      SELECT unique_id,
        max(feature = 'acc_jci'  AND status = 'found') AS has_jci,
        max(feature = 'acc_nabh' AND status = 'found') AS has_nabh,
        max(feature = 'teaching_text' AND status = 'found') AS teach_txt,
        sum(CASE WHEN feature IN ('xray','usg','ct','mri','pet','mammography') AND status = 'found' THEN 1 ELSE 0 END) AS n_diag,
        first(CASE WHEN feature = 'acc_jci' AND status = 'found' THEN evidence END, true)  AS ev_jci,
        first(CASE WHEN feature = 'acc_nabh' AND status = 'found' THEN evidence END, true) AS ev_nabh,
        first(CASE WHEN feature = 'teaching_text' AND status = 'found' THEN evidence END, true) AS ev_teach
      FROM text_feats GROUP BY unique_id),
    beds_txt AS (SELECT unique_id, max(qty) AS beds_parsed,
                   first(CASE WHEN qty IS NOT NULL THEN array(named_struct('strength', strength, 'field', field, 'text', text)) END, true) AS ev
                 FROM {hits_view} WHERE feature = 'beds' GROUP BY unique_id),
    w AS (SELECT f.*, h.has_jci, h.has_nabh, h.teach_txt, h.n_diag, h.ev_jci, h.ev_nabh, h.ev_teach,
                 b.beds_parsed, b.ev AS ev_beds, coalesce(f.capacity, f.capacity_text, b.beds_parsed) AS beds
          FROM {clean_view} f LEFT JOIN h ON h.unique_id = f.unique_id LEFT JOIN beds_txt b ON b.unique_id = f.unique_id),
    structured AS (
      SELECT unique_id, 'facility_type' AS feature,
        CASE facility_type WHEN 'hospital' THEN 1.0 WHEN 'nursing_home' THEN 0.7 WHEN 'clinic' THEN 0.4 WHEN 'doctor' THEN 0.2 END AS value,
        CASE WHEN facility_type IS NULL THEN 'unknown' ELSE 'found' END AS status, CAST(NULL AS DOUBLE) AS qty, false AS found_in_text,
        CAST(NULL AS ARRAY<STRUCT<strength: DOUBLE, field: STRING, text: STRING>>) AS evidence FROM w
      UNION ALL SELECT unique_id, 'doctors', CASE WHEN n_doctors IS NOT NULL THEN least(1.0, ln(1 + n_doctors) / ln(101)) END,
        CASE WHEN n_doctors IS NOT NULL THEN 'found' ELSE 'unknown' END, n_doctors, false, NULL FROM w
      UNION ALL SELECT unique_id, 'beds', CASE WHEN beds IS NOT NULL THEN least(1.0, ln(1 + beds) / ln(1001)) END,
        CASE WHEN beds IS NOT NULL THEN 'found' ELSE 'unknown' END, beds,
        capacity IS NULL AND capacity_text IS NULL AND beds_parsed IS NOT NULL,
        CASE WHEN capacity IS NULL AND capacity_text IS NULL THEN ev_beds END FROM w
      UNION ALL SELECT unique_id, 'accreditation',
        CASE WHEN has_jci THEN 1.0 WHEN has_nabh THEN 0.8 WHEN has_text AND NOT truncated THEN 0.0 END,
        CASE WHEN has_jci OR has_nabh THEN 'found' WHEN NOT has_text THEN 'unknown' WHEN truncated THEN 'truncated' ELSE 'not_mentioned' END,
        NULL, coalesce(has_jci OR has_nabh, false), coalesce(ev_jci, ev_nabh) FROM w
      UNION ALL SELECT unique_id, 'teaching',
        CASE WHEN array_contains(coalesce(affiliations, CAST(array() AS ARRAY<STRING>)), 'academic') OR coalesce(teach_txt, false)
                  OR lower(name) rlike 'medical college|institute of medical sciences|aiims' THEN 1.0
             WHEN affiliations IS NOT NULL OR has_text THEN 0.0 END,
        'derived', NULL, coalesce(teach_txt, false), ev_teach FROM w
      UNION ALL SELECT unique_id, 'diagnostic_breadth', CASE WHEN has_text THEN coalesce(n_diag, 0) / 6.0 END,
        'derived', CAST(n_diag AS DOUBLE), false, NULL FROM w
      UNION ALL SELECT unique_id, 'years_operating',
        CASE WHEN year_established IS NOT NULL THEN least(1.0, ln(1 + greatest(0, {REF_YEAR} - year_established)) / ln(101)) END,
        CASE WHEN year_established IS NOT NULL THEN 'found' ELSE 'unknown' END, CAST({REF_YEAR} - year_established AS DOUBLE), false, NULL FROM w
      UNION ALL SELECT unique_id, 'is_public',
        CASE WHEN operator_type IN ('public', 'government') OR array_contains(coalesce(affiliations, CAST(array() AS ARRAY<STRING>)), 'government') THEN 1.0
             WHEN operator_type IS NOT NULL THEN 0.0 END, 'derived', NULL, false, NULL FROM w
      UNION ALL SELECT unique_id, 'obstetric_focus',                                          -- G16 (R4)
        CASE WHEN name_type = 'maternity' THEN 1.0
             WHEN arrays_overlap(coalesce(specialties_arr, CAST(array() AS ARRAY<STRING>)), array('obstetrics', 'maternalFetalMedicine')) THEN 0.6 ELSE 0.0 END,
        'derived', NULL, false, NULL FROM w)
    SELECT unique_id, feature, CAST(value AS DOUBLE) AS value, status, CAST(qty AS DOUBLE) AS qty, found_in_text, evidence FROM text_feats
    WHERE feature NOT IN ('acc_jci', 'acc_nabh', 'teaching_text')
    UNION ALL
    SELECT unique_id, feature, CAST(value AS DOUBLE), status, CAST(qty AS DOUBLE), found_in_text, evidence FROM structured""")

if REBUILD:
    with step("4 silver_feature_hits"):
        save("silver_feature_hits", hits_df(TBL["silver_claims"], TBL["silver_facility_clean"]))
    with step("4 gold_facility_features"):
        save("gold_facility_features", features_df(TBL["silver_feature_hits"], TBL["silver_facility_clean"]))

## 6. Step 6: `gold_facility_completeness` (SCORING §2)

In [0]:
if REBUILD:
    with step("6 gold_facility_completeness"):
        df = spark.sql(f"""
        WITH b AS (SELECT unique_id, value IS NOT NULL AS beds_known FROM {TBL['gold_facility_features']} WHERE feature = 'beds'),
        x AS (
          SELECT f.*, l.location_quality, coalesce(b.beds_known, false) AS beds_known,
            try_to_timestamp(substr(f.last_updated, 1, 10), 'yyyy-MM-dd') AS upd,
            (f.official_phone IS NOT NULL OR coalesce(size(f.phone_numbers), 0) > 0) AS has_phone
          FROM {TBL['silver_facility_clean']} f
          JOIN {TBL['silver_facility_location']} l ON l.unique_id = f.unique_id
          LEFT JOIN b ON b.unique_id = f.unique_id),
        s AS (
          SELECT unique_id, location_quality,
            CASE location_quality WHEN 'exact' THEN 25 WHEN 'approximate' THEN 15
                 ELSE CASE WHEN city IS NOT NULL OR state IS NOT NULL THEN 5 ELSE 0 END END
            + CASE WHEN has_phone THEN 10 ELSE 0 END
            + CASE WHEN n_doctors IS NOT NULL THEN 10 ELSE 0 END
            + CASE WHEN beds_known THEN 10 ELSE 0 END
            + CASE WHEN procedure_arr IS NOT NULL THEN 10 ELSE 0 END
            + CASE WHEN equipment_arr IS NOT NULL THEN 10 ELSE 0 END
            + CASE WHEN capability_arr IS NOT NULL THEN 10 ELSE 0 END
            + CASE WHEN year_established IS NOT NULL THEN 5 ELSE 0 END
            + CASE WHEN upd IS NULL THEN 0 WHEN months_between(current_date(), upd) <= 24 THEN 10 ELSE 5 END AS score,
            filter(array(
              CASE WHEN location_quality = 'approximate' THEN 'exact location (only the PIN area is known)'
                   WHEN location_quality = 'unknown' THEN 'location' END,
              CASE WHEN NOT has_phone THEN 'phone number' END,
              CASE WHEN n_doctors IS NULL THEN 'number of doctors' END,
              CASE WHEN NOT beds_known THEN 'number of beds' END,
              CASE WHEN procedure_arr IS NULL THEN 'list of procedures' END,
              CASE WHEN equipment_arr IS NULL THEN 'list of equipment' END,
              CASE WHEN capability_arr IS NULL THEN 'list of capabilities' END,
              CASE WHEN year_established IS NULL THEN 'year established' END,
              CASE WHEN upd IS NULL THEN 'date the record was last updated' END), x -> x IS NOT NULL) AS missing_fields
          FROM x)
        SELECT unique_id, CAST(score AS INT) AS completeness_score,
          CASE WHEN score >= 70 THEN 'rich' WHEN score >= 40 THEN 'partial' ELSE 'sparse' END AS completeness_band,
          CASE location_quality WHEN 'exact' THEN 'exact' WHEN 'approximate' THEN 'pin' ELSE 'none' END AS location_precision,
          missing_fields
        FROM s""")
        save("gold_facility_completeness", df)

## 7. Step 7: `gold_service_evidence` (SCORING §3.2–3.4 + X1, X2, X6, R1, R2)

Pure Python, run per facility with `mapInPandas`. The regression fixtures (§9) call exactly the same `evaluate_facility`.

In [0]:
FIELD_W = {"procedure": 1.0, "equipment": 1.0, "capability": 0.9, "description": 0.6}
FIELD_RANK = {"procedure": 0, "equipment": 1, "capability": 2, "description": 3}
# "quantity": a number attached to a count/size word ("10 units", "20-bed", "3T", "64-slice"); not "24-hour"/"24x7"/"60,000"
QTY_RX = (r"(?<![\d,.])\b\d+(?:\.\d+)?\s*-?\s*(?:bedded|beds?|units?|machines?|stations?|chairs?|slices?|tesla|t\b|"
          r"modular|ots?\b|operation theat|operating theat|rooms?|ventilators?|incubators?|warmers?|cots?|"
          r"(?:dialys|haemodialys|hemodialys)\w*)")
GENERIC_RX = r"departments? (?:include|such as)|specialit(?:y|ies) (?:include|such as)|services (?:include|such as)|including"
MRI_STRONG_ITEM = r"tesla|\b\d(?:\.\d)?\s*-?\s*t\b|scanner"
CLINIC_HOURS_SERVICES = {"emergency_csection", "nicu", "trauma"}
LABEL_RANK = {"Strong": 0, "Moderate": 1, "Weak": 2, "Possible": 3, "No evidence": 4, "Not offered": 5}
_RX = {}

def rx(p):
    if p not in _RX:
        _RX[p] = re.compile(p)
    return _RX[p]

def _found(feats, f, text_only=False):
    x = feats.get(f) or {}
    return x.get("status") == "found" and (not text_only or bool(x.get("txt")))

def _raw_items(raw):
    if raw is None:
        return []
    try:
        v = json.loads(raw)
        return [x for x in v if isinstance(x, str)] if isinstance(v, list) else [str(v)]
    except Exception:
        return [raw]

def _specialty_quote(fac, codes):
    """A verbatim item from the raw specialties field that names one of these codes (bench checks the raw field)."""
    for exact in (True, False):          # prefer the item that IS the code, then one that contains it
        for code in codes:
            if code in fac["specs"]:
                for item in fac["raw_spec_items"]:
                    if (item.lower() == code.lower()) if exact else (code.lower() in item.lower()):
                        return {"field": "specialties", "quote": item}
    return None

def evaluate_service(fac, svc):
    d = SVC[svc]
    specs, feats, claims = fac["specs"], fac["feats"], fac["claims"]
    codes_present = [c for c in d["codes"] if c in specs]
    cond_codes_ok = any(c in specs for c in d.get("cond_codes", []))
    pos, neg, eq_only = [], [], []
    for c in claims:
        lc = c["lc"]
        if d.get("excl") and rx(d["excl"]).search(lc):
            continue                                                                      # exclusion, same clause
        tested = rx(d["strip"]).sub(" ", lc) if d.get("strip") else lc                     # §3.2 step 4
        hit = rx(d["direct"]).search(tested) or (
            d.get("cond") and rx(d["cond"]).search(tested)
            and (rx(d["cond_item"]).search(c["item_lc"]) or cond_codes_ok))               # X1 / X7
        if hit:
            (neg if c["neg"] else pos).append(c)
        elif d.get("strip") and not c["neg"] and c["field"] in ("procedure", "equipment") and rx(d["strip"]).search(lc):
            eq_only.append(c)
    eq_claim = False
    if not pos and eq_only:                                                                # equipment-only claim at ×0.8
        pos, eq_claim = eq_only, True

    def weight(c):
        spec = 1.2 if rx(QTY_RX).search(c["lc"]) else (0.6 if (rx(GENERIC_RX).search(c["lc"]) or c["lc"].count(",") >= 3) else 1.0)
        return FIELD_W.get(c["field"], 0.6) * spec * (0.8 if eq_claim else 1.0), spec

    # X6: dedup at clause level; a clause contained in an already-kept clause is the same claim
    groups = []
    for c in sorted(pos, key=lambda c: (-len(c["norm"] or ""), FIELD_RANK.get(c["field"], 9), c["pos"], c["cpos"])):
        if not c["norm"]:
            continue
        g = next((g for g in groups if c["norm"] in g["norm"]), None)
        (g["members"] if g else groups.append({"norm": c["norm"], "members": []}) or groups[-1]["members"]).append(c)
    kept = []
    for g in groups:
        best = max(g["members"], key=lambda c: (weight(c)[0], -FIELD_RANK.get(c["field"], 9)))
        kept.append(dict(best, w=weight(best)[0], qty=weight(best)[1] == 1.2))
    kept.sort(key=lambda c: (-c["w"], FIELD_RANK.get(c["field"], 9), c["pos"], c["cpos"]))
    w_best = kept[0]["w"] if kept else 0.0

    # service-specific equipment (non-negated text, optional same-item context)
    equip_claim = None
    if d.get("equip"):
        for c in sorted(claims, key=lambda c: (FIELD_RANK.get(c["field"], 9), c["pos"], c["cpos"])):
            if not c["neg"] and rx(d["equip"]).search(c["lc"]) and (not d.get("equip_ctx") or rx(d["equip_ctx"]).search(c["item_lc"])):
                if d.get("excl") and rx(d["excl"]).search(c["lc"]):
                    continue
                equip_claim = c
                break
    has_equip = equip_claim is not None

    corro = []
    if codes_present: corro.append("specialty code")
    if has_equip and not eq_claim: corro.append("service-specific equipment")
    if len(kept) >= 2: corro.append("second distinct claim")

    nontext = " ".join(c["lc"] for c in claims if not c["neg"])
    in_text = {
        "ot": _found(feats, "ot"), "anaesthesia": _found(feats, "anaesthesia"), "blood_bank": _found(feats, "blood_bank"),
        "icu_ccu": _found(feats, "icu") or _found(feats, "ccu"), "icu": _found(feats, "icu"),
        "emergency_24x7": _found(feats, "emergency_24x7"),
        "nephrology": "nephrology" in specs or "nephrologist" in nontext,
        "neo_code": bool({"neonatology", "pediatrics"} & specs), "radiology_code": "radiology" in specs,
        "onco_code": bool(set(SVC["oncology"]["codes"]) & specs), "ent_code": "otolaryngology" in specs,
    }
    defined = d["implied"] + d["checked"]
    met, pre_met, pre_missing = 0.0, [], []
    for p in defined:
        if (kept and p in d["implied"]) or in_text[p]:
            met += 1; pre_met.append(PREREQ_NAMES[p])
        elif fac["truncated"]:
            met += 0.5
            if p in d["checked"]: pre_missing.append(PREREQ_NAMES[p] + " (not reported, list cut off)")
        elif p in d["checked"]:
            pre_missing.append(PREREQ_NAMES[p])
        elif p in d["implied"] and not kept:
            pass                                              # implied prerequisites are never shown as missing
    pts_direct = 40 * min(w_best, 1.0)
    pts_corro = 10 * min(len(corro), 3)
    pts_pre = 20 * met / len(defined) if defined else 0.0
    pts_qty = 10 if any(k["qty"] for k in kept) else 0
    E = pts_direct + pts_corro + pts_pre + pts_qty if kept else min(50.0, pts_corro + pts_pre)
    E = round(E, 1)

    # ---- Strong rule per service (SCORING §3.1 "Strong also needs", R1) ----
    beds = (feats.get("beds") or {}).get("qty")
    nt = fac["name_type"]
    if svc == "dialysis":
        rule, why_not = has_equip or nt == "nephrology_dialysis", "no dialysis equipment and not a kidney/dialysis centre by name"
    elif svc == "emergency_csection":                                                     # R1
        g5 = _found(feats, "emergency_24x7", text_only=True)
        support = (_found(feats, "labour_room") or _found(feats, "ot") or nt == "maternity"
                   or bool({"obstetrics", "maternalFetalMedicine"} & specs))
        rule = g5 and support
        why_not = "emergency readiness not confirmed" if not g5 else "no labour room, operating theatre, obstetrics specialty or maternity focus"
    elif svc == "nicu":
        rule, why_not = _found(feats, "neonatal_support"), "no neonatal equipment (ventilator, CPAP, warmer, incubator or phototherapy)"
    elif svc == "cardiac_surgery":
        rule = has_equip and not (beds is not None and beds < 30)
        why_not = "no cath lab, CTICU or heart-lung machine" if not has_equip else "fewer than 30 beds"
    elif svc == "mri":
        rule = any(k["field"] == "equipment" or rx(MRI_STRONG_ITEM).search(k["item_lc"]) for k in kept)
        why_not = "MRI not listed as equipment or with a tesla/scanner detail"
    elif svc == "trauma":
        rule, why_not = has_equip, "no CT scanner or ambulance"
    elif svc == "burn_care":
        rule, why_not = has_equip, "no burn-specific equipment (burn ICU, dressing room)"
    elif svc == "oncology":
        rule, why_not = has_equip or nt == "oncology", "no LINAC, PET-CT or brachytherapy, and not a cancer centre by name"
    else:
        rule, why_not = has_equip, "no audiometry, BERA or ENT endoscope"
    # row 4 claim condition with R2
    field_ok = any(k["field"] in ("procedure", "equipment") for k in kept) or (
        d.get("specialist") is not None and nt == d["specialist"]
        and any(k["field"] in ("description", "capability") and k["qty"] for k in kept))

    caveats = []
    if neg and not kept:
        label = "Not offered"
    elif not kept:
        label = "Possible" if (codes_present or has_equip) else "No evidence"
    elif neg:
        label = "Weak"; caveats.append("The record has conflicting statements about this service")
    elif E >= 70 and field_ok and rule:
        label = "Strong"
    elif E >= 45:
        label = "Moderate"
        if E >= 70:
            caveats.append(why_not[0].upper() + why_not[1:] if not rule else "The claim appears only in the description or capability text")
    else:
        label = "Weak"
    if svc == "emergency_csection" and kept and not _found(feats, "emergency_24x7", text_only=True):
        if "Emergency readiness not confirmed" not in caveats:
            caveats.append("Emergency readiness not confirmed")                           # R1
    if svc in CLINIC_HOURS_SERVICES and kept and _found(feats, "emergency_24x7") and _found(feats, "clinic_hours"):
        caveats.append("The record also lists clinic hours: call to confirm emergency cover")   # R5
    if fac["truncated"] and label not in ("No evidence", "Not offered"):
        caveats.append("Some lists in this record were cut off at 50 items, so evidence may be missing")

    # ---- why: verbatim quotes, primary claim first ----
    why = []
    def add(field, text):
        if text and len(why) < 4 and all(w["quote"] != text for w in why):
            why.append({"field": field, "quote": text})
    if kept:
        add(kept[0]["field"], kept[0]["text"])
    if svc == "emergency_csection" and kept:             # R1 evidence: the emergency-readiness quote explains a Strong label
        x = feats.get("emergency_24x7") or {}
        if x.get("status") == "found" and x.get("qfield") and x["qfield"] != "specialties":
            add(x["qfield"], x.get("quote"))
    if equip_claim is not None:
        add(equip_claim["field"], equip_claim["text"])
    for k in kept[1:2]:
        add(k["field"], k["text"])
    sq = _specialty_quote(fac, codes_present)
    if sq:
        add(sq["field"], sq["quote"])
    for p in ("blood_bank", "ot", "emergency_24x7", "icu"):
        if p in defined and len(why) < 4:
            x = feats.get(p) or {}
            if x.get("status") == "found" and x.get("qfield") and x["qfield"] != "specialties":
                add(x["qfield"], x.get("quote"))
    if not kept and neg:
        add(neg[0]["field"], neg[0]["text"])
    # quote check: every quote must be verbatim in the facility's raw field
    bad = 0
    for w in why:
        src = fac["raw"].get(w["field"], [])
        if not any(w["quote"] in s for s in src):
            bad += 1

    return dict(
        unique_id=fac["unique_id"], service=svc, evidence_score=float(E), evidence_label=label,
        caveats_json=json.dumps(caveats), primary_field=kept[0]["field"] if kept else None,
        primary_quote=kept[0]["text"] if kept else None,
        supporting_claims_json=json.dumps([{"field": k["field"], "claim_text": k["text"], "weight": round(k["w"], 3)} for k in kept]),
        absence_claims_json=json.dumps([{"field": c["field"], "claim_text": c["text"]} for c in neg]),
        prerequisites_met_json=json.dumps(pre_met), prerequisites_missing_json=json.dumps(pre_missing),
        strong_rule_met=bool(rule) if kept else False, strong_rule_missing=None if (rule or not kept) else why_not,
        has_direct_or_code=bool(kept) or bool(codes_present), n_claims=len(kept), n_negated=len(neg),
        why_json=json.dumps(why), quotes_not_in_raw=bad,
        e_breakdown=(f"direct claim {pts_direct:.0f} + corroboration {pts_corro} ({', '.join(corro) or 'none'}) + "
                     f"prerequisites {pts_pre:.0f} + quantity {pts_qty}") if kept else
                    f"no direct claim: corroboration {pts_corro} + prerequisites {pts_pre:.0f} (capped at 50)")

def evaluate_facility(fac):
    return [evaluate_service(fac, s) for s in SVC]

EVIDENCE_SCHEMA = T.StructType([
    T.StructField("unique_id", T.StringType()), T.StructField("service", T.StringType()),
    T.StructField("evidence_score", T.DoubleType()), T.StructField("evidence_label", T.StringType()),
    T.StructField("caveats_json", T.StringType()), T.StructField("primary_field", T.StringType()),
    T.StructField("primary_quote", T.StringType()), T.StructField("supporting_claims_json", T.StringType()),
    T.StructField("absence_claims_json", T.StringType()), T.StructField("prerequisites_met_json", T.StringType()),
    T.StructField("prerequisites_missing_json", T.StringType()), T.StructField("strong_rule_met", T.BooleanType()),
    T.StructField("strong_rule_missing", T.StringType()), T.StructField("has_direct_or_code", T.BooleanType()),
    T.StructField("n_claims", T.IntegerType()), T.StructField("n_negated", T.IntegerType()),
    T.StructField("why_json", T.StringType()), T.StructField("quotes_not_in_raw", T.IntegerType()),
    T.StructField("e_breakdown", T.StringType())])

def fac_from_row(r):
    """Turn one row of evidence_input_df into the dict evaluate_facility expects."""
    claims = json.loads(r["claims_json"]) if r["claims_json"] else []
    raw = {"description": [r["raw_description"]] if r["raw_description"] else [],
           "procedure": _raw_items(r["raw_procedure"]), "equipment": _raw_items(r["raw_equipment"]),
           "capability": _raw_items(r["raw_capability"]), "specialties": _raw_items(r["raw_specialties"])}
    specs = set(json.loads(r["spec_json"])) if r["spec_json"] else set()
    return dict(unique_id=r["unique_id"], name_type=r["name_type"], truncated=bool(r["truncated"]), specs=specs,
                claims=claims, feats=json.loads(r["feats_json"]) if r["feats_json"] else {}, raw=raw,
                raw_spec_items=raw["specialties"])

def evidence_input_df(clean_view, claims_view, features_view):
    return spark.sql(f"""
    SELECT f.unique_id, f.name_type, f.truncated, to_json(f.specialties_arr) AS spec_json,
      f.raw_description, f.raw_procedure, f.raw_equipment, f.raw_capability, f.raw_specialties,
      c.claims_json, ff.feats_json
    FROM {clean_view} f
    LEFT JOIN (SELECT unique_id, to_json(collect_list(named_struct('field', field, 'pos', pos, 'cpos', cpos, 'text', text,
                 'lc', lc, 'item_lc', item_lc, 'norm', norm_clause, 'neg', is_negated))) AS claims_json
               FROM {claims_view} GROUP BY unique_id) c ON c.unique_id = f.unique_id
    LEFT JOIN (SELECT unique_id, to_json(map_from_entries(collect_list(struct(feature, named_struct(
                 'status', status, 'value', value, 'qty', qty, 'txt', found_in_text,
                 'quote', try_element_at(evidence, 1).text, 'qfield', try_element_at(evidence, 1).field))))) AS feats_json
               FROM {features_view} GROUP BY unique_id) ff ON ff.unique_id = f.unique_id""")

def _evidence_partition(batches):
    import pandas as pd
    for pdf in batches:
        rows = []
        for r in pdf.to_dict("records"):
            rows.extend(evaluate_facility(fac_from_row(r)))
        yield pd.DataFrame(rows, columns=[f.name for f in EVIDENCE_SCHEMA.fields])

if REBUILD:
    with step("7 gold_service_evidence"):
        ev = evidence_input_df(TBL["silver_facility_clean"], TBL["silver_claims"], TBL["gold_facility_features"]) \
            .repartition(16).mapInPandas(_evidence_partition, EVIDENCE_SCHEMA)
        ev.createOrReplaceTempView("ev_raw")
        df = spark.sql("""
        SELECT unique_id, service, evidence_score, evidence_label,
          from_json(caveats_json, 'array<string>') AS caveats,
          CASE WHEN primary_quote IS NOT NULL THEN named_struct('field', primary_field, 'claim_text', primary_quote) END AS primary_quote,
          from_json(supporting_claims_json, 'array<struct<field:string,claim_text:string,weight:double>>') AS supporting_claims,
          from_json(absence_claims_json, 'array<struct<field:string,claim_text:string>>') AS absence_claims,
          from_json(prerequisites_met_json, 'array<string>') AS prerequisites_met,
          from_json(prerequisites_missing_json, 'array<string>') AS prerequisites_missing,
          strong_rule_met, strong_rule_missing, has_direct_or_code, n_claims, n_negated,
          why_json, quotes_not_in_raw, e_breakdown
        FROM ev_raw""")
        save("gold_service_evidence", df)
    print(q(f"SELECT sum(quotes_not_in_raw) AS bad_quotes, count(*) AS cells FROM {TBL['gold_service_evidence']}").collect()[0].asDict())

## 8. Step 5 (run after evidence, see notes): `gold_service_depth` with v5 weights (R4) and the R3 truncation rule

In [0]:
if REBUILD:
    with step("5 gold_service_depth"):
        s_list = ", ".join(f"'{s}'" for s in S_FEATURES)
        df = spark.sql(f"""
        WITH x AS (
          SELECT ff.unique_id, w.service, w.feature, w.weight, ff.value, ff.status, ff.qty, ff.evidence,
            CASE WHEN ff.status = 'truncated' AND w.feature IN ({s_list})                       -- R3
                 THEN CASE WHEN e.has_direct_or_code THEN 0.5 ELSE 0.0 END
                 ELSE coalesce(ff.value, 0.5) END AS v
          FROM {TBL['ref_service_weights']} w
          JOIN {TBL['gold_facility_features']} ff ON ff.feature = w.feature
          JOIN {TBL['gold_service_evidence']} e ON e.unique_id = ff.unique_id AND e.service = w.service)
        SELECT unique_id, service,
          100 * sum(weight * v) / sum(weight)                                              AS depth_D,
          sum(CASE WHEN value IS NOT NULL THEN weight ELSE 0 END) / sum(weight)            AS feature_coverage,
          slice(array_sort(collect_list(named_struct('contrib', weight * v, 'feature', feature, 'status', status,
                'qty', qty, 'evidence', evidence)),
                (a, b) -> CASE WHEN a.contrib > b.contrib THEN -1 WHEN a.contrib < b.contrib THEN 1 ELSE 0 END), 1, 4) AS top_contributors,
          filter(collect_list(CASE WHEN value IS NULL THEN feature END), z -> z IS NOT NULL) AS unknown_features
        FROM x GROUP BY unique_id, service""")
        save("gold_service_depth", df)

## 9. Step 8: `gold_service_scores` (everything that doesn't depend on the query)

In [0]:
if REBUILD:
    with step("8 gold_service_scores"):
        gate_sql = " ".join(f"WHEN '{s}' THEN {LABEL_RANK[p[1]]}" for s, p in PROFILE.items())
        df = spark.sql(f"""
        WITH lr AS (SELECT * FROM VALUES {", ".join(f"('{k}', {v})" for k, v in LABEL_RANK.items())} AS t(label, r))
        SELECT d.unique_id, d.service, f.name, f.facility_type,
          e.evidence_score AS E, e.evidence_label, d.depth_D AS D, c.completeness_score AS C, c.completeness_band,
          lr.r <= CASE d.service {gate_sql} END AND e.evidence_label NOT IN ('Possible', 'No evidence', 'Not offered') AS passes_gate,
          e.evidence_label = 'Not offered' AS not_offered,
          CASE WHEN c.completeness_band = 'sparse' OR d.feature_coverage < 0.4
                    OR (l.location_quality <> 'exact' AND e.evidence_label <> 'Strong') THEN 'Low'
               WHEN e.evidence_label = 'Strong' AND c.completeness_band = 'rich' AND d.feature_coverage >= 0.6 THEN 'High'
               ELSE 'Medium' END AS confidence,
          d.feature_coverage, d.top_contributors, d.unknown_features,
          e.primary_quote, e.supporting_claims, e.prerequisites_missing, e.caveats, e.why_json, e.e_breakdown,
          e.strong_rule_met, e.strong_rule_missing, c.missing_fields,
          l.lat, l.lon, l.location_quality, l.pin_radius_km,
          coalesce(f.official_phone, try_element_at(f.phone_numbers, 1)) AS official_phone,
          slice(filter(array_distinct(concat(array(f.official_website), coalesce(f.websites, CAST(array() AS ARRAY<STRING>)))), z -> z IS NOT NULL), 1, 3) AS references,
          f.city, f.district, f.state
        FROM {TBL['gold_service_depth']} d
        JOIN {TBL['gold_service_evidence']} e ON e.unique_id = d.unique_id AND e.service = d.service
        JOIN {TBL['gold_facility_completeness']} c ON c.unique_id = d.unique_id
        JOIN {TBL['silver_facility_location']} l ON l.unique_id = d.unique_id
        JOIN {TBL['silver_facility_clean']} f ON f.unique_id = d.unique_id
        JOIN lr ON lr.label = e.evidence_label""")
        save("gold_service_scores", df)

## 10. Regression fixtures (PLAN_V5 §6): hard assertions. The build stops if any fails.

In [0]:
FIXTURES = [  # case_id, text, field, target, expected
    ("F1", "Performs cardiac bypass surgery in the Cardiac Cath Lab", "procedure", "cardiac_surgery.direct", "found"),
    ("F2", "Laparoscopic gastric bypass surgery", "procedure", "cardiac_surgery.direct", "not found"),
    ("F3", "24-hour dialysis services with 10 units", "description", "S1.dialysis_unit", "found, qty 10"),
    ("F4", "Non-NABH accreditation", "capability", "G10.accreditation", "not found"),
    ("F5", "24x7 nursing", "capability", "G5.emergency_24x7", "not found"),
    ("F6", "24x7 Pharmacy", "capability", "G5.emergency_24x7", "not found"),
    ("F7", "Offers 24/7 emergency care", "capability", "G5.emergency_24x7", "found"),
    ("F8", "Offers emergency and elective services", "capability", "G5.emergency_24x7", "found"),
    ("F9", "Heart Institute for cardiac care", "description", "S6.ccu", "not found"),
    ("F10", "Intensive Critical Care Unit (ICCU)", "capability", "S6.ccu / G4.icu", "S6 not found, G4 found"),
    ("F11", "Dialysis not available", "capability", "dialysis.direct", "negated"),
    ("F12", "Largest dialysis centre in the region", "description+capability", "dialysis.second_claim", "no (deduplicated)"),
]
save("ref_regression_fixtures", spark.createDataFrame(FIXTURES, "case_id STRING, text STRING, field STRING, target STRING, expected STRING"))

def fixture_facility(case_id, text, field):
    fields = field.split("+")
    arr = lambda f: [text] if f in fields else None
    return (f"fx_{case_id}", f"Fixture {case_id}", None, None, None, None, None, None, None, None,
            arr("procedure"), arr("equipment"), arr("capability"), text if "description" in fields else None,
            True, False, None, None, None, None, None, None, None, None, None, None, None, None,
            text if "description" in fields else None,
            json.dumps([text]) if "procedure" in fields else None, json.dumps([text]) if "equipment" in fields else None,
            json.dumps([text]) if "capability" in fields else None, None)

clean_schema = spark.table(TBL["silver_facility_clean"]).schema
fx_clean = spark.createDataFrame([fixture_facility(c, t, f) for c, t, f, _, _ in FIXTURES], clean_schema)
fx_clean.createOrReplaceTempView("fx_clean")
claims_df("fx_clean").createOrReplaceTempView("fx_claims")
hits_df("fx_claims", "fx_clean").createOrReplaceTempView("fx_hits")
features_df("fx_hits", "fx_clean").createOrReplaceTempView("fx_features")
fx_feat = {(r.unique_id, r.feature): r for r in spark.table("fx_features").collect()}
fx_ev = {}
for r in evidence_input_df("fx_clean", "fx_claims", "fx_features").collect():
    for e in evaluate_facility(fac_from_row(r.asDict())):
        fx_ev[(r.unique_id, e["service"])] = e

def fstatus(cid, feat):
    r = fx_feat.get((f"fx_{cid}", feat))
    return (r.status == "found" and r.found_in_text, r.qty) if r else (False, None)

checks = {
    "F1": fx_ev[("fx_F1", "cardiac_surgery")]["n_claims"] > 0,
    "F2": fx_ev[("fx_F2", "cardiac_surgery")]["n_claims"] == 0,
    "F3": fstatus("F3", "dialysis_unit") == (True, 10.0),
    "F4": not fstatus("F4", "accreditation")[0],
    "F5": not fstatus("F5", "emergency_24x7")[0],
    "F6": not fstatus("F6", "emergency_24x7")[0],
    "F7": fstatus("F7", "emergency_24x7")[0],
    "F8": fstatus("F8", "emergency_24x7")[0],
    "F9": not fstatus("F9", "ccu")[0],
    "F10": (not fstatus("F10", "ccu")[0]) and fstatus("F10", "icu")[0],
    "F11": fx_ev[("fx_F11", "dialysis")]["n_negated"] > 0 and fx_ev[("fx_F11", "dialysis")]["evidence_label"] == "Not offered",
    "F12": fx_ev[("fx_F12", "dialysis")]["n_claims"] == 1,
}
FIXTURE_RESULTS = [(c, t, tg, exp, "PASS" if checks[c] else "FAIL") for c, t, _, tg, exp in FIXTURES]
for row in FIXTURE_RESULTS:
    print(f"  {row[4]}  {row[0]:4s} {row[2]:24s} expected {row[3]:24s} | {row[1]}")
assert all(checks.values()), f"Regression fixtures failed: {[c for c, ok in checks.items() if not ok]}"
print("All 12 regression fixtures pass.")

## 11. `find_care(service, location, k)`: query time (SCORING §4, PLAN_V5 §5)

In [0]:
import numpy as np
import pandas as pd

HOW_RANKED_LEAD = {
    "dialysis": "Dialysis is needed several times a week, so we show the closest facilities whose own records say they provide dialysis",
    "emergency_csection": "For emergency delivery we show the closest facilities whose records say they perform caesareans",
    "nicu": "For a newborn needing intensive care we show the closest facilities whose records describe a NICU",
    "trauma": "For injuries every minute counts, so we show the closest facilities whose records describe trauma or accident care",
    "cardiac_surgery": "For heart surgery we prioritise equipment and experience over distance, because it's often worth travelling",
    "oncology": "For cancer treatment we prioritise treatment capability (radiotherapy, chemotherapy) over distance",
    "mri": "For an MRI scan we show nearby facilities whose records list an MRI",
    "burn_care": "For burns we look for a documented burn unit, which is worth some extra travel",
    "ent": "For ear, nose and throat care, which is widely available, we balance distance, evidence and the number of doctors",
}

def how_ranked(service):
    tc, gate, wE, wD, wA, dh, mx = PROFILE[service]
    parts = sorted([(wA, "closeness"), (wE, "how clearly the record documents it"), (wD, "equipment and capability")], reverse=True)
    weights = ", ".join(f"{int(round(w * 100))}% {n}" for w, n in parts)
    return f"{HOW_RANKED_LEAD[service]}, ranked {weights}, searching up to {mx} km."

_CACHE = {}
SCORE_COLS = ["unique_id", "service", "name", "facility_type", "E", "evidence_label", "D", "C", "completeness_band", "passes_gate",
              "not_offered", "confidence", "feature_coverage", "why_json", "caveats", "prerequisites_missing", "missing_fields",
              "lat", "lon", "location_quality", "pin_radius_km", "official_phone", "references", "city", "district", "state",
              "e_breakdown", "strong_rule_missing"]

def _scores(service):
    if service not in _CACHE:
        pdf = spark.table(TBL["gold_service_scores"]).where(F.col("service") == service).select(*SCORE_COLS).toPandas()
        _CACHE[service] = pdf[pdf["lat"].notna() & pdf["lon"].notna()].reset_index(drop=True)
    return _CACHE[service]

def _geo():
    if "geo" not in _CACHE:
        pins = q(f"""SELECT pin, lat, lon, lat_pre, lon_pre, coord_outlier, district, state FROM {PINS_OK}""").toPandas()
        # every located facility; `exact` marks the ones with their own coordinates (the rest sit at their PIN centre)
        cities = spark.sql(f"""SELECT f.city, f.state, l.lat, l.lon, l.location_quality = 'exact' AS exact
                               FROM {TBL['silver_facility_clean']} f
                               JOIN {TBL['silver_facility_location']} l ON l.unique_id = f.unique_id
                               WHERE f.city IS NOT NULL AND l.lat IS NOT NULL AND l.lon IS NOT NULL""").toPandas()
        for df_, col in ((pins, "district"), (pins, "state"), (cities, "city"), (cities, "state")):
            df_[col + "_n"] = df_[col].fillna("").map(_norm)
        _CACHE["geo"] = (pins, cities)
    return _CACHE["geo"]

def _norm(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    s = re.sub(r"\b(district|dist|city|nagar nigam)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def _title(s):
    return " ".join(w.capitalize() for w in str(s).lower().split())

def resolve_location(text):
    """→ (resolved dict | None, clarification dict | None, error str | None)."""
    t = (text or "").strip()
    m = re.fullmatch(r"(-?\d{1,2}(?:\.\d+)?)\s*,\s*(-?\d{1,3}(?:\.\d+)?)", t)
    if m:
        lat, lon = float(m.group(1)), float(m.group(2))
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            return None, None, "Those coordinates are out of range."
        return {"lat": lat, "lon": lon, "method": "latlon", "label": f"{lat:.4f}, {lon:.4f}"}, None, None
    pins, cities = _geo()
    if re.fullmatch(r"\d{6}", t.replace(" ", "")):
        row = pins[pins["pin"] == float(t.replace(" ", ""))]
        r = None if row.empty else row.iloc[0]
        if r is not None and bool(r["coord_outlier"]) and not pd.isna(r["lat_pre"]):
            # the directory's point for this PIN is in the wrong place: use the centre of its 3-digit PIN area instead
            return {"lat": float(r["lat_pre"]), "lon": float(r["lon_pre"]), "method": "pin",
                    "label": f"PIN {t.replace(' ', '')} ({_title(r['district'])}, {_title(r['state'])}; approximate, "
                             f"centre of PIN area {t.replace(' ', '')[:3]}xxx)"}, None, None
        if r is None or pd.isna(r["lat"]):
            return None, None, f"PIN code {t} is not in the PIN directory. Try a nearby PIN, the district or the city."
        return {"lat": float(r["lat"]), "lon": float(r["lon"]), "method": "pin",
                "label": f"PIN {t.replace(' ', '')} ({_title(r['district'])}, {_title(r['state'])})"}, None, None
    # "name" or "name, state"
    name, state = (t.split(",", 1) + [""])[:2] if "," in t else (t, "")
    name_n, state_n = _norm(name), _norm(state)
    if not name_n:
        return None, None, "Please type a PIN code, district, city or 'lat,lon'."
    d = pins[pins["district_n"] == name_n]
    c_all = cities[cities["city_n"] == name_n]
    if state_n:
        d = d[d["state_n"].str.contains(state_n, regex=False)]
        c_all = c_all[c_all["state_n"].str.contains(state_n, regex=False)]
    c = c_all[c_all["exact"].astype(bool)]
    # a state counts for a city only if it holds ≥ 2 facilities and ≥ 5% of that city's facilities (stray records aren't a "city")
    cc = c[c["state_n"] != ""].groupby("state_n").size()
    city_states = set(cc[(cc >= 2) & (cc >= 0.05 * cc.sum())].index) if len(cc) else set()
    states = sorted(set(d.loc[d["state_n"] != "", "state_n"]) | city_states)
    if not states:
        # Small towns (eval bench: Tiuni, Palsana, Kayamkulam): only one listed facility, or only PIN-centre locations.
        # Use any located facility with that city name; several states → ask which one.
        small = c_all[c_all["state_n"] != ""]
        small_states = sorted(set(small["state_n"]))
        if len(small_states) == 1:
            n_ = len(small)
            return {"lat": float(small["lat"].median()), "lon": float(small["lon"].median()), "method": "city",
                    "label": f"{_title(name)}, {_title(small_states[0])} (approximate: centre of {n_} listed "
                             f"facilit{'y' if n_ == 1 else 'ies'} in this town)"}, None, None
        if len(small_states) > 1:
            return None, {"message": f"'{_title(name)}' exists in more than one state. Which one do you mean?",
                          "options": [{"label": f"{_title(name)}, {_title(s)}", "state": _title(s), "kind": "city"}
                                      for s in small_states]}, None
        return None, None, f"We couldn't find '{t}'. Try a 6-digit PIN code, a district name, a city name, or 'lat,lon'."
    if len(states) > 1:
        opts = [{"label": f"{_title(name)}, {_title(s)}", "state": _title(s),
                 "kind": "city" if s in city_states else "district"} for s in states]
        return None, {"message": f"'{_title(name)}' exists in more than one state. Which one do you mean?",
                      "options": opts}, None
    st = states[0]
    cs, ds = c[c["state_n"] == st], d[(d["state_n"] == st) & d["lat"].notna()]   # outlier PIN points excluded
    if len(cs) >= 3:   # a city with facilities: median of that city's facilities (SCORING §4.3)
        return {"lat": float(cs["lat"].median()), "lon": float(cs["lon"].median()), "method": "city",
                "label": f"{_title(name)} city, {_title(st)} (centre of {len(cs)} listed facilities)"}, None, None
    if len(ds):
        # median, not mean: a few PINs carry stray coordinates (Patna has one at lat 20.3), which drag a mean ~35 km off
        return {"lat": float(ds["lat"].median()), "lon": float(ds["lon"].median()), "method": "district",
                "label": f"{_title(name)} district, {_title(st)} (centre of {len(ds)} PIN area{'s' if len(ds) != 1 else ''})"}, None, None
    if cs.empty:
        return None, None, f"We found '{t}' but have no reliable coordinates for it. Try a 6-digit PIN code or 'lat,lon'."
    return {"lat": float(cs["lat"].median()), "lon": float(cs["lon"].median()), "method": "city",
            "label": f"{_title(name)} city, {_title(st)}"}, None, None

def _haversine(lat, lon, lats, lons):
    la1, lo1, la2, lo2 = map(np.radians, (lat, lon, lats, lons))
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

def _jl(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return []
    if isinstance(v, str):
        return json.loads(v)
    return [x.asDict() if hasattr(x, "asDict") else x for x in list(v)]

def _ref_url(u):
    u = str(u).strip()
    return u if re.match(r"https?://", u) else "https://" + u

# Silver's record-contamination flags. The plan has no rule for them (it assumes clean data), so they add a caveat
# (stating uncertainty) and never change the score or rank.
MIXED_FLAGS = {"flag_capability_mentions_other_facility": "capabilities mention another facility",
               "flag_description_other_state": "the description names a different state"}

def _mixed():
    if "mixed" not in _CACHE:
        cols = [c for c in MIXED_FLAGS if c in sf_types]
        out = {}
        if cols:
            for row in spark.sql(f"SELECT unique_id, {', '.join(cols)} FROM {SILVER} WHERE {' OR '.join(cols)}").collect():
                out[row["unique_id"]] = [MIXED_FLAGS[c] for c in cols if row[c]]
        _CACHE["mixed"] = out
    return _CACHE["mixed"]

def _result(r, rank, service):
    caveats = list(_jl(r["caveats"]))
    mixed = _mixed().get(r["unique_id"])
    if mixed:
        caveats.append("This record may include text from another facility (" + " and ".join(mixed) + "). Call to confirm.")
    unc = float(r["pin_radius_km"]) if r["location_quality"] != "exact" else 0.0
    if r["location_quality"] != "exact":
        caveats.append(f"Location is approximate (centre of the PIN area), so the distance may be off by about {unc:.0f} km")
    if r["completeness_band"] == "sparse":
        caveats.append("Limited information. Call to confirm before travelling.")
    return {
        "rank": rank, "unique_id": r["unique_id"], "name": r["name"], "facility_type": r["facility_type"],
        "distance_km": round(float(r["road_km"]), 1), "distance_uncertainty_km": round(unc, 1),
        "evidence_label": r["evidence_label"], "confidence": r["confidence"],
        "rank_score": round(float(r["score"]), 1), "E": round(float(r["E"]), 1), "D": round(float(r["D"]), 1),
        "A": round(float(r["A"]), 1),
        "why": _jl(r["why_json"]),
        "caveats": caveats,
        "not_reported": list(_jl(r["missing_fields"])) + list(_jl(r["prerequisites_missing"])),
        "contact": {"official_phone": r["official_phone"] if isinstance(r["official_phone"], str) else None,
                    "references": [_ref_url(u) for u in _jl(r["references"])][:3]},
    }

def find_care(service, location, k=5, restrict_ids=None, log=True):
    """Return the PLAN_V5 §5 response dict. restrict_ids limits candidates (pilot fixture only; not logged)."""
    service = (service or "").strip()
    k = max(1, min(int(k), 20))
    request = {"service": service, "location": location, "k": k}
    resp = {"query": {"service": service, "location_input": location, "resolved_location": None, "radius_km_used": None},
            "how_ranked": None, "results": [], "extra_rows": None, "fallback": None}
    if service not in PROFILE:
        resp["error"] = f"Unknown service '{service}'. Choose one of: {', '.join(SERVICES)}."
        return _finish(request, resp, log and restrict_ids is None)
    loc, clar, err = resolve_location(location)
    if clar:
        resp["clarification"] = clar
        return _finish(request, resp, log and restrict_ids is None)
    if err:
        resp["error"] = err
        return _finish(request, resp, log and restrict_ids is None)
    tc, gate_min, wE, wD, wA, d_half, max_r = PROFILE[service]
    resp["query"]["resolved_location"] = loc
    resp["how_ranked"] = how_ranked(service)

    df = _scores(service)
    if restrict_ids is not None:
        df = df[df["unique_id"].isin(set(restrict_ids))]
    df = df.copy()
    df["road_km"] = 1.4 * _haversine(loc["lat"], loc["lon"], df["lat"].to_numpy(float), df["lon"].to_numpy(float))
    unc = np.where(df["location_quality"] != "exact", df["pin_radius_km"].astype(float), 0.0)
    df["A"] = 100 * np.power(0.5, (df["road_km"] + unc) / d_half)                    # approximate location: add PIN radius
    df["score"] = wE * df["E"] + wD * df["D"] + wA * df["A"]
    df["lrank"] = df["evidence_label"].map(LABEL_RANK)
    order = ["score", "lrank", "road_km", "unique_id"]
    asc = [False, True, True, True]
    gate = df[df["passes_gate"] & ~df["not_offered"]]

    used = max_r
    for r_ in [s for s in RADIUS_STEPS if s < max_r] + [max_r]:
        if (gate["road_km"] <= r_).sum() >= k:
            used = r_
            break
    resp["query"]["radius_km_used"] = used
    top = gate[gate["road_km"] <= used].sort_values(order, ascending=asc).head(k)
    resp["results"] = [_result(r, i, service) for i, (_, r) in enumerate(top.iterrows(), 1)]

    extra = {}
    if not (top["evidence_label"] == "Strong").any():
        strong = gate[(gate["evidence_label"] == "Strong") & (gate["road_km"] <= max_r) & ~gate["unique_id"].isin(top["unique_id"])]
        if len(strong):
            s = strong.sort_values(order, ascending=asc).iloc[0]
            extra["stronger_evidence"] = {"message": f"Stronger documented evidence, {s['road_km']:.0f} km away",
                                          "result": _result(s, None, service)}
    if not tc and len(top) < k:
        poss = df[(df["evidence_label"] == "Possible") & (df["road_km"] <= max_r)].sort_values(order, ascending=asc).head(k - len(top))
        if len(poss):
            extra["unconfirmed_options"] = {
                "message": "Unconfirmed options: these records only hint at this service (a specialty or equipment), with no direct claim. Call first.",
                "results": [_result(r, None, service) for _, r in poss.iterrows()]}
    resp["extra_rows"] = extra or None

    if not resp["results"]:
        msg = (f"No facility with good evidence for {SERVICE_PHRASE[service]} within {max_r} km of {loc['label']}. "
               "Call before travelling.")
        if len(gate):
            n = gate.sort_values("road_km").iloc[0]
            msg += f" The nearest facility whose record documents it is {n['name']}, about {n['road_km']:.0f} km away (road estimate)."
        resp["fallback"] = msg
    return _finish(request, resp, log and restrict_ids is None)

LOG_FQ = TBL["gold_query_log"]
if not table_exists(LOG_FQ):
    spark.sql(f"CREATE TABLE {LOG_FQ} (query_id STRING, ts TIMESTAMP, request_json STRING, response_json STRING)"
              + ("" if IS_DATABRICKS else " USING parquet"))
    spark.sql(f"ALTER TABLE {LOG_FQ} SET TBLPROPERTIES ('ftrc.owner' = '{OWNER_TAG}')")
LOG_SCHEMA = "query_id STRING, ts TIMESTAMP, request_json STRING, response_json STRING"

def _finish(request, resp, log):
    if log:
        row = [(str(uuid.uuid4()), dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
                json.dumps(request, ensure_ascii=False), json.dumps(resp, ensure_ascii=False))]
        spark.createDataFrame(row, LOG_SCHEMA).write.mode("append").saveAsTable(LOG_FQ)
    return resp

# ---------------------------------------------------------------- rendering ------------------------------------------------
LABEL_COLOR = {"Strong": "#2d6a4f", "Moderate": "#9c6500", "Weak": "#8a4b08", "Possible": "#6c757d"}
FIELD_NAME = {"procedure": "procedures", "equipment": "equipment", "capability": "capabilities", "description": "description",
              "specialties": "specialties"}

def _card(r, service):
    e = html.escape
    dist = f"~{r['distance_km']:.0f} km (road est.)" + (f" ± {r['distance_uncertainty_km']:.0f} km" if r["distance_uncertainty_km"] else "")
    why = "".join(f"<div>✓ Virtue Foundation record, {e(FIELD_NAME.get(w['field'], w['field']))}: “{e(w['quote'])}”</div>" for w in r["why"])
    nr = "".join(f"<div>? {e(x)}</div>" for x in r["not_reported"])
    cv = "".join(f"<div>⚠ {e(x)}</div>" for x in r["caveats"])
    refs = " · ".join(f"<a href='{e(u)}' target='_blank'>{e(re.sub(r'^https?://', '', u))}</a>" for u in r["contact"]["references"])
    phone = e(r["contact"]["official_phone"] or "not reported")
    rank = f"{r['rank']}. " if r["rank"] else ""
    return f"""<div style="border:1px solid #ddd;border-radius:8px;padding:10px 14px;margin:8px 0;font-family:sans-serif;font-size:14px">
      <div style="font-size:16px"><b>{rank}{e(r['name'])}</b> &nbsp; {e(dist)} · {e(SERVICE_NAMES[service])}:
        <b style="color:{LABEL_COLOR.get(r['evidence_label'], '#333')}">{e(r['evidence_label'].upper())}</b> · Confidence: <b>{e(r['confidence'])}</b></div>
      <div style="margin-top:6px"><b>Why:</b>{why or '<div>(no quote)</div>'}</div>
      {f'<div style="margin-top:6px;color:#8a4b08"><b>Caveats:</b>{cv}</div>' if cv else ''}
      {f'<div style="margin-top:6px;color:#555"><b>Not reported:</b>{nr}</div>' if nr else ''}
      <div style="margin-top:6px;color:#333">Score: Evidence {r['E']:.0f} · Capability {r['D']:.0f} · Access {r['A']:.0f}
        &nbsp; | &nbsp; Listed in dataset. Call to confirm: <b>{phone}</b>{(' · ' + refs) if refs else ''}</div></div>"""

def render_html(resp):
    e = html.escape
    qy, svc = resp["query"], resp["query"]["service"]
    out = [f"<div style='font-family:sans-serif'><h3 style='margin:4px 0'>{e(SERVICE_NAMES.get(svc, svc))} near {e(str(qy['location_input']))}</h3>"]
    if resp.get("error"):
        return out[0] + f"<p style='color:#b00'>{e(resp['error'])}</p></div>"
    if resp.get("clarification"):
        c = resp["clarification"]
        return out[0] + f"<p><b>{e(c['message'])}</b></p><ul>" + "".join(f"<li>{e(o['label'])}</li>" for o in c["options"]) + \
            "</ul><p>Type the location again with the state, for example <i>" + e(c["options"][0]["label"]) + "</i>.</p></div>"
    loc = qy["resolved_location"]
    out.append(f"<p style='color:#555;margin:2px 0'>Searched around {e(loc['label'])}, within {qy['radius_km_used']} km.</p>")
    out.append(f"<p><b>How we ranked:</b> {e(resp['how_ranked'])}</p>")
    for r in resp["results"]:
        out.append(_card(r, svc))
    if resp.get("fallback"):
        out.append(f"<p style='color:#b00'><b>{e(resp['fallback'])}</b></p>")
    ex = resp.get("extra_rows") or {}
    if "stronger_evidence" in ex:
        out.append(f"<h4>{e(ex['stronger_evidence']['message'])}</h4>" + _card(ex["stronger_evidence"]["result"], svc))
    if "unconfirmed_options" in ex:
        out.append(f"<h4>{e(ex['unconfirmed_options']['message'])}</h4>" + "".join(_card(r, svc) for r in ex["unconfirmed_options"]["results"]))
    out.append("<p style='color:#777;font-size:12px'>We measure what each facility's own record says, not whether the service is available today. "
               "Confidence is about how much the record tells us. Always call before travelling.</p></div>")
    return "".join(out)

def render_text(resp):
    qy = resp["query"]
    lines = [f"{SERVICE_NAMES.get(qy['service'], qy['service'])} near {qy['location_input']}"]
    if resp.get("error"):
        return "\n".join(lines + ["  ERROR: " + resp["error"]])
    if resp.get("clarification"):
        return "\n".join(lines + ["  " + resp["clarification"]["message"]] + [f"   - {o['label']}" for o in resp["clarification"]["options"]])
    lines.append(f"  resolved: {qy['resolved_location']['label']} ({qy['resolved_location']['method']}), radius {qy['radius_km_used']} km")
    lines.append(f"  how ranked: {resp['how_ranked']}")
    def one(r):
        out = [f"  {r['rank'] or '-'}. {r['name']}  ~{r['distance_km']} km · {r['evidence_label'].upper()} · Confidence: {r['confidence']}"
               f" · score {r['rank_score']} (E {r['E']:.0f} · D {r['D']:.0f} · A {r['A']:.0f})"]
        out += [f"       ✓ {w['field']}: \"{w['quote'][:140]}\"" for w in r["why"]]
        out += [f"       ⚠ {c}" for c in r["caveats"]]
        if r["not_reported"]:
            out.append("       ? " + " · ".join(r["not_reported"]))
        return out
    for r in resp["results"]:
        lines += one(r)
    if resp.get("fallback"):
        lines.append("  FALLBACK: " + resp["fallback"])
    for key_, v in (resp.get("extra_rows") or {}).items():
        lines.append(f"  [{key_}] {v['message']}")
        for r in ([v["result"]] if "result" in v else v["results"]):
            lines += one(r)
    return "\n".join(lines)

def show(resp):
    if IS_DATABRICKS:
        displayHTML(render_html(resp))  # noqa: F821
    else:
        print(render_text(resp))

## Your search
Pick a **service**, type a **location** at the top of the notebook, then run this cell.

In [0]:
resp = find_care(W["service"], W["location"], int(W["k"]))
show(resp)

## 12. SQL-callable version: `find_care_sql(service, location, k)`
Location here is a 6-digit PIN or `lat,lon`. Districts, cities and ambiguity handling are in the Python `find_care`.

In [0]:
SQL_FN = None
if IS_DATABRICKS:
    for cand in ("find_care_sql", "find_care_sql_v5"):
        fq = f"{SCHEMA}.{cand}"
        try:
            info = " ".join(str(r[0]) for r in spark.sql(f"DESCRIBE FUNCTION EXTENDED {fq}").collect())
            if OWNER_TAG not in info:
                continue           # exists, not ours
        except Exception:
            pass                   # doesn't exist
        SQL_FN = fq
        break
    steps_sql = ", ".join(f"{float(s)}D" for s in RADIUS_STEPS)
    try:
        spark.sql(f"""
        CREATE OR REPLACE FUNCTION {SQL_FN}(p_service STRING, p_location STRING, k INT)
        RETURNS TABLE (rank INT, unique_id STRING, name STRING, distance_km DOUBLE, evidence_label STRING, confidence STRING,
                       rank_score DOUBLE, E DOUBLE, D DOUBLE, A DOUBLE, primary_quote STRING, radius_km_used DOUBLE)
        COMMENT 'ftrc.owner={OWNER_TAG}: PIN or lat,lon only; see notebook find_care for districts, cities and the full response'
        RETURN
        WITH loc AS (
          SELECT lat, lon FROM (
            SELECT coalesce(lat, lat_pre) AS lat, coalesce(lon, lon_pre) AS lon, 1 AS o FROM {PINS_OK}
            WHERE p_location rlike '^ *[0-9]{{6}} *$' AND pin = try_cast(trim(p_location) AS DOUBLE)
            UNION ALL
            SELECT try_cast(trim(get(split(p_location, ','), 0)) AS DOUBLE), try_cast(trim(get(split(p_location, ','), 1)) AS DOUBLE), 2
            WHERE p_location rlike '^ *-?[0-9.]+ *, *-?[0-9.]+ *$') ORDER BY o LIMIT 1),
        p AS (SELECT * FROM {TBL['ref_service_profile']} WHERE service = p_service),
        c AS (
          SELECT s.*, p.wE, p.wD, p.wA, p.d_half_km, p.max_radius_km,
            1.4 * 2 * 6371 * asin(sqrt(pow(sin(radians(s.lat - loc.lat) / 2), 2)
              + cos(radians(loc.lat)) * cos(radians(s.lat)) * pow(sin(radians(s.lon - loc.lon) / 2), 2))) AS road_km
          FROM {TBL['gold_service_scores']} s CROSS JOIN loc CROSS JOIN p
          WHERE s.service = p_service AND s.lat IS NOT NULL AND s.passes_gate AND NOT s.not_offered),
        steps AS (SELECT DISTINCT r FROM (SELECT explode(concat(filter(array({steps_sql}), x -> x < p.max_radius_km), array(p.max_radius_km))) AS r FROM p)),
        cnt AS (SELECT steps.r, count_if(c.road_km <= steps.r) AS n FROM steps LEFT JOIN c ON true GROUP BY steps.r),
        -- smallest step with ≥ k results, else the largest step (written as ORDER BY: Databricks rejects an aggregate
        -- mixing the outer parameter k with local columns in a correlated predicate)
        used AS (SELECT r FROM cnt ORDER BY CASE WHEN n >= k THEN 0 ELSE 1 END, CASE WHEN n >= k THEN r ELSE -r END LIMIT 1),
        sc AS (
          SELECT c.*, used.r AS r_used, 100 * pow(0.5, (c.road_km + CASE WHEN c.location_quality <> 'exact' THEN c.pin_radius_km ELSE 0 END) / c.d_half_km) AS A_
          FROM c CROSS JOIN used WHERE c.road_km <= used.r),
        ranked AS (
          SELECT sc.*, wE * E + wD * D + wA * A_ AS score,
            row_number() OVER (ORDER BY wE * E + wD * D + wA * A_ DESC, road_km, unique_id) AS rn FROM sc)
        SELECT CAST(rn AS INT), unique_id, name, round(road_km, 1), evidence_label, confidence, round(score, 1),
               round(E, 1), round(D, 1), round(A_, 1), primary_quote.claim_text, r_used
        FROM ranked WHERE rn <= k ORDER BY rn""")
        print("SQL function:", SQL_FN)
        display(spark.sql(f"SELECT * FROM {SQL_FN}('dialysis', '492001', 5)"))  # noqa: F821
    except Exception as ex:
        note(f"SQL table function not created: {str(ex)[:300]}")

## 13. End-to-end pilot fixture (PLAN_V5 §6): 15 Raipur facilities × 3 README queries

In [0]:
PILOT_IDS = {
    "CARE": "723a6e17-fdf9-4af8-ae80-b16ea3b98244", "SMC": "0b02d119-a409-4554-9979-39817ed38e3e",
    "MMI": "c35dabaa-544a-4d93-9405-b5444ed0bab3", "Shri Krishna": "8e0dbec3-fd46-42e1-b213-9204bf799f69",
    "Kanha": "509c6668-953b-4d13-b198-5659718b54e6", "Suyash": "3d4c1b68-7402-427c-922c-adbf52bd7dab",
    "Tiwari": "f73efb65-a436-45a4-b6b3-8409e412525d", "Devi": "6c08e1f3-1d59-42c0-bb0c-4617f95a8986",
    "Kanwar": "3933c2cf-fae4-439e-a72d-39d2c28ddf4e", "Vasudev": "702a22a9-0c4c-49b4-afbe-e54b42750283",
    "Vidya": "78bed667-b16e-43c1-8d82-c82891f23aa8", "AIIMS": "e43a21d0-8c78-436d-9656-39418b1fbf16",
    "Mamta": "f71b47e9-1504-40f9-9256-706f6173280b", "Sanjeevani": "269e3a30-c0d8-447c-8d74-fe6a2977a315",
    "Gayatri": "8f5bea35-e441-441f-ae15-cca315bbee5b"}
ID2SHORT = {v: k for k, v in PILOT_IDS.items()}
PILOT_Q = {"Q1": ("dialysis", "21.23855,81.64300"), "Q2": ("emergency_csection", "21.23784,81.70542"),
           "Q3": ("cardiac_surgery", "21.25457,81.65708")}
PILOT = {}
if W["run_tests"] == "yes":
    for qid, (svc, loc) in PILOT_Q.items():
        PILOT[qid] = find_care(svc, loc, 15, restrict_ids=list(PILOT_IDS.values()), log=False)
        print(f"--- {qid} {svc}")
        for r in PILOT[qid]["results"]:
            print(f"  {r['rank']:2d}. {ID2SHORT.get(r['unique_id'], r['unique_id'])[:14]:14s} {r['evidence_label']:9s} "
                  f"score {r['rank_score']:5.1f}  E {r['E']:5.1f}  D {r['D']:5.1f}  A {r['A']:5.1f}  {r['confidence']}")
    top = lambda qid, n: [ID2SHORT.get(r["unique_id"]) for r in PILOT[qid]["results"][:n]]
    PILOT_CHECKS = {
        "Q1 top 3 = {CARE, Vidya, Shri Krishna}": set(top("Q1", 3)) == {"CARE", "Vidya", "Shri Krishna"},
        "Q3 top 2 ⊇ {Suyash, CARE}": {"Suyash", "CARE"} <= set(top("Q3", 2)),
        "Q2 top 3 ⊇ {Devi, Kanwar}": {"Devi", "Kanwar"} <= set(top("Q2", 3)),
        "Q2 has ≥ 1 Strong C-section": any(r["evidence_label"] == "Strong" for r in PILOT["Q2"]["results"]),
    }
    for k_, ok in PILOT_CHECKS.items():
        print(("  PASS  " if ok else "  FAIL  ") + k_)

## 14. Smoke tests (logged to `gold_query_log`)

In [0]:
SMOKE = [
    ("dialysis", "492001", "urban PIN, recurring"),
    ("emergency_csection", "Patna", "city, time-critical"),
    ("nicu", "110001", "metro PIN, time-critical"),
    ("trauma", "Barmer", "rural district, time-critical"),
    ("cardiac_surgery", "Chennai", "city, travel-worthy"),
    ("mri", "Jalandhar", "city, non-critical"),
    ("oncology", "Guwahati", "north-east city"),
    ("ent", "26.85,80.95", "lat,lon (Lucknow)"),
    ("burn_care", "Bhopal", "beta service"),
    ("dialysis", "Dantewada", "rural district (tribal Chhattisgarh)"),
    ("trauma", "744301", "care-desert candidate (Car Nicobar PIN)"),
    ("emergency_csection", "Aurangabad", "ambiguous city (Maharashtra / Bihar)"),
    ("nicu", "Bilaspur", "ambiguous city (Chhattisgarh / Himachal)"),
    ("emergency_csection", "Aurangabad, Bihar", "disambiguated"),
]
SMOKE_OUT = []
if W["run_tests"] == "yes":
    for svc, loc, why_ in SMOKE:
        t0 = time.time()
        r = find_care(svc, loc, 5)
        SMOKE_OUT.append((svc, loc, why_, r, round(time.time() - t0, 2)))
        print(f"\n### {svc} @ {loc} [{why_}] ({SMOKE_OUT[-1][4]} s)")
        print(render_text(r))

## 15. Label and confidence counts per service (all facilities)

In [0]:
LABEL_COUNTS = spark.sql(f"""
SELECT service,
  count_if(evidence_label = 'Strong') AS strong, count_if(evidence_label = 'Moderate') AS moderate,
  count_if(evidence_label = 'Weak') AS weak, count_if(evidence_label = 'Possible') AS possible,
  count_if(evidence_label = 'Not offered') AS not_offered, count_if(evidence_label = 'No evidence') AS no_evidence,
  count_if(passes_gate) AS passes_gate,
  count_if(passes_gate AND confidence = 'High') AS gate_high, count_if(passes_gate AND confidence = 'Medium') AS gate_medium,
  count_if(passes_gate AND confidence = 'Low') AS gate_low,
  round(percentile_approx(D, 0.5), 1) AS median_D, round(avg(feature_coverage), 2) AS avg_coverage
FROM {TBL['gold_service_scores']} GROUP BY service ORDER BY service""")
LABEL_ROWS = [r.asDict() for r in LABEL_COUNTS.collect()]
if IS_DATABRICKS:
    display(LABEL_COUNTS)  # noqa: F821
else:
    for r in LABEL_ROWS:
        print(r)

## 16. Environment checks: challenge brief / output format in the workspace, `ai_query` availability

In [0]:
ENV = {}
try:
    ENV["volumes"] = [r.asDict() for r in spark.sql(f"SHOW VOLUMES IN {SCHEMA}").collect()]
except Exception as ex:
    ENV["volumes"] = f"n/a: {str(ex)[:120]}"
try:
    ENV["tables"] = [r[1] for r in spark.sql(f"SHOW TABLES IN {SCHEMA}").collect()]
except Exception as ex:
    ENV["tables"] = f"n/a: {str(ex)[:120]}"
if IS_DATABRICKS:
    found = []
    for v in (ENV["volumes"] if isinstance(ENV["volumes"], list) else []):
        name = v.get("volume_name") or list(v.values())[-1]
        path = f"/Volumes/{SCHEMA.replace('.', '/')}/{name}"
        try:
            stack = [path]
            while stack and len(found) < 200:
                for fi in dbutils.fs.ls(stack.pop()):  # noqa: F821
                    (stack.append(fi.path) if fi.isDir() else found.append(fi.path))
        except Exception as ex:
            found.append(f"{path}: {str(ex)[:80]}")
    ENV["volume_files"] = found
    try:
        from databricks.sdk import WorkspaceClient
        wc = WorkspaceClient()
        hits = []
        for root in ("/Workspace/Shared", "/Shared"):
            try:
                for o in wc.workspace.list(root, recursive=True):
                    if re.search(r"brief|readme|challenge|format|submission|rubric", o.path, re.I):
                        hits.append(o.path)
            except Exception:
                pass
        ENV["workspace_brief_candidates"] = hits[:50]
    except Exception as ex:
        ENV["workspace_brief_candidates"] = f"n/a: {str(ex)[:120]}"
    try:
        spark.sql("DESCRIBE FUNCTION ai_query").collect()
        ENV["ai_query_function"] = "registered"
        try:
            ENV["ai_query_call"] = spark.sql("SELECT ai_query('databricks-meta-llama-3-3-70b-instruct', 'Reply with OK') AS r").collect()[0][0][:40]
        except Exception as ex:
            ENV["ai_query_call"] = f"failed: {str(ex)[:200]}"
    except Exception as ex:
        ENV["ai_query_function"] = f"not available: {str(ex)[:120]}"
print(json.dumps(ENV, indent=1, default=str)[:4000])

## 17. Run summary

In [0]:
RUN["total_s"] = round(time.time() - RUN["t0"], 1)
print("Step timings (s):", json.dumps(RUN["steps"], indent=1))
print("Total:", RUN["total_s"], "s")
print("\nTables:", json.dumps(TBL, indent=1))
print("\nAdaptations:")
for a in ADAPTATIONS:
    print(" -", a)
SUMMARY = {"tables": TBL, "steps_s": RUN["steps"], "total_s": RUN["total_s"], "adaptations": ADAPTATIONS,
           "fixtures": FIXTURE_RESULTS, "pilot_checks": globals().get("PILOT_CHECKS"), "labels": LABEL_ROWS, "env": ENV,
           "smoke": [(s, l, w, render_text(r), t) for s, l, w, r, t in SMOKE_OUT]}
if IS_DATABRICKS:
    _lite = dict(SUMMARY, smoke=[(s, l, w, txt[:6000], t) for s, l, w, txt, t in SUMMARY["smoke"]],
                 pilot={k: render_text(v) for k, v in PILOT.items()})
    dbutils.notebook.exit(json.dumps(_lite, default=str)[:900000])  # noqa: F821